In [1]:
#/////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////
#                                              Import Packages                                                                       #
#/////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////

import boto3
import json
import pyodbc
import pandas as pd
from datetime import date, datetime, timedelta
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np
import plotly.express as px
import mplcursors
import plotly.graph_objects as go
from pandas.tseries.holiday import USFederalHolidayCalendar
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import RidgeCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import TimeSeriesSplit

In [2]:
#/////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////
#                                                          Set TODAY                                                                 #
#/////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////

# For automation use:
# TODAY = pd.Timestamp(date.today())

# For backtesting, set manually:
TODAY = pd.Timestamp("2026-03-10")
TODAY = pd.to_datetime(TODAY)
print("Today's date is:", TODAY)

Today's date is: 2026-03-10 00:00:00


In [ ]:
#/////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////
#                                     Connect to SQL Server: Load in Historical Data                                                 #
#/////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////

def get_secret(secret_name, region_name):
    """
    Retrieve secrets from AWS Secrets Manager.
    """
    try:
        session = boto3.session.Session()
        client  = session.client(service_name="secretsmanager", region_name=region_name)
        response = client.get_secret_value(SecretId=secret_name)
        secret   = json.loads(response["SecretString"])
        return secret
    except Exception as e:
        print(f"Failed to retrieve secret: {e}")
        return None

secret_name = "analytics/prod/credentials"
region_name = "us-east-1"

secret = get_secret(secret_name, region_name)

if secret:
    try:
        server   = secret["datamart"]["host"]
        port     = secret["datamart"]["port"]
        database = secret["datamart"]["db"]
        username = secret["datamart"]["user"]
        password = secret["datamart"]["password"]

        conn_str = (
            f"Driver={{ODBC Driver 17 for SQL Server}};"
            f"Server={server},{port};"
            f"Database={database};"
            f"UID={username};"
            f"PWD={password};"
        )

        select_query = """
SET DATEFIRST 1;  -- Monday = 1

WITH channels AS (
    SELECT DISTINCT
        CASE
            WHEN channel LIKE '%corr%' THEN 'Correspondent'
            ELSE channel
        END AS channel
    FROM marketing_sandbox.dbo.SDS WITH (NOLOCK)
    WHERE channel NOT LIKE '%Credit Union Partners%'
),

base AS (
    SELECT
        CASE
            WHEN s.channel LIKE '%corr%' THEN 'Correspondent'
            ELSE s.channel
        END AS channel,
        s.funded_date,
        s.loan_amount,
        DATEPART(YEAR, s.funded_date) AS year_of_funding,
        DATEPART(WEEKDAY, s.funded_date) - 1 AS day_of_week,
        DATEPART(DAY, s.funded_date) AS day_of_month,
        DATEDIFF(WEEK, DATEADD(MONTH, DATEDIFF(MONTH, 0, s.funded_date), 0), s.funded_date) + 1 AS week_of_month,
        DATEPART(MONTH, s.funded_date) AS month_of_year
    FROM marketing_sandbox.dbo.SDS s WITH (NOLOCK)
    WHERE s.funded_date >= DATEADD(YEAR, -5, CAST(GETDATE() AS DATE))
      AND s.funded_date <= (
            SELECT MIN(Calendar_Date)
            FROM (
                SELECT Calendar_Date,
                       ROW_NUMBER() OVER (ORDER BY Calendar_Date ASC) AS rn
                FROM marketing_sandbox.dbo.Calendar WITH (NOLOCK)
                WHERE Calendar_Date > CAST(GETDATE() AS DATE)
                  AND Biz_Day = 1
            ) biz
            WHERE rn = 3
      )
      AND s.funded_date IS NOT NULL
      AND s.channel NOT LIKE '%Credit Union Partners%'
),

freq AS (
    SELECT
        channel,
        funded_date,
        loan_amount,
        year_of_funding,
        day_of_week,
        day_of_month,
        week_of_month,
        month_of_year,
        COUNT(CASE WHEN day_of_week < 5 THEN 1 END)
            OVER (PARTITION BY channel, year_of_funding, month_of_year, week_of_month) AS loans_in_week,
        COUNT(CASE WHEN day_of_week < 5 THEN 1 END)
            OVER (PARTITION BY channel, year_of_funding, month_of_year) AS loans_in_month,
        COUNT(CASE WHEN day_of_week < 5 THEN 1 END)
            OVER (PARTITION BY channel, year_of_funding, month_of_year, week_of_month, day_of_week) AS loans_on_day_of_week,
        COUNT(CASE WHEN day_of_week < 5 THEN 1 END)
            OVER (PARTITION BY channel, year_of_funding, month_of_year, day_of_month) AS loans_on_day_of_month,
        SUM(CASE WHEN day_of_week < 5 THEN loan_amount ELSE 0 END)
            OVER (PARTITION BY channel, year_of_funding, month_of_year, week_of_month) AS amount_in_week,
        SUM(CASE WHEN day_of_week < 5 THEN loan_amount ELSE 0 END)
            OVER (PARTITION BY channel, year_of_funding, month_of_year) AS amount_in_month,
        SUM(CASE WHEN day_of_week < 5 THEN loan_amount ELSE 0 END)
            OVER (PARTITION BY channel, year_of_funding, month_of_year, week_of_month, day_of_week) AS amount_on_day_of_week,
        SUM(CASE WHEN day_of_week < 5 THEN loan_amount ELSE 0 END)
            OVER (PARTITION BY channel, year_of_funding, month_of_year, day_of_month) AS amount_on_day_of_month,
        SUM(CASE WHEN day_of_week < 5 THEN 1 ELSE 0 END)
            OVER (PARTITION BY channel, month_of_year) AS total_loans_in_month_across_years,
        SUM(CASE WHEN day_of_week < 5 THEN loan_amount ELSE 0 END)
            OVER (PARTITION BY channel, month_of_year) AS total_amount_in_month_across_years,
        SUM(CASE WHEN day_of_week < 5 THEN 1 ELSE 0 END)
            OVER (PARTITION BY channel, year_of_funding) AS total_loans_in_year,
        SUM(CASE WHEN day_of_week < 5 THEN loan_amount ELSE 0 END)
            OVER (PARTITION BY channel, year_of_funding) AS total_amount_in_year
    FROM base
),

agg_loans AS (
    SELECT
        channel,
        CAST(funded_date AS DATE) AS funded_date,
        year_of_funding,
        day_of_week,
        day_of_month,
        week_of_month,
        month_of_year,
        COUNT(*) AS number_of_loans,
        SUM(loan_amount) AS total_loan_amount,
        CAST(loans_in_week AS FLOAT) / NULLIF(loans_in_month, 0) AS week_weight,
        CAST(loans_on_day_of_week AS FLOAT) / NULLIF(loans_in_week, 0) AS day_of_week_weight,
        CAST(loans_on_day_of_month AS FLOAT) / NULLIF(loans_in_month, 0) AS day_of_month_weight,
        CAST(amount_in_week AS FLOAT) / NULLIF(amount_in_month, 0) AS week_amount_weight,
        CAST(amount_on_day_of_week AS FLOAT) / NULLIF(amount_in_week, 0) AS day_of_week_amount_weight,
        CAST(amount_on_day_of_month AS FLOAT) / NULLIF(amount_in_month, 0) AS day_of_month_amount_weight,
        CAST(total_loans_in_month_across_years AS FLOAT) / NULLIF(SUM(total_loans_in_month_across_years)
            OVER (PARTITION BY channel, month_of_year), 0) AS month_to_month_weight_loans,
        CAST(total_amount_in_month_across_years AS FLOAT) / NULLIF(SUM(total_amount_in_month_across_years)
            OVER (PARTITION BY channel, month_of_year), 0) AS month_to_month_weight_amount,
        CAST(total_loans_in_month_across_years AS FLOAT) / NULLIF(total_loans_in_year, 0) AS month_within_year_weight_loans,
        CAST(total_amount_in_month_across_years AS FLOAT) / NULLIF(total_amount_in_year, 0) AS month_within_year_weight_amount
    FROM freq
    GROUP BY
        channel, CAST(funded_date AS DATE), year_of_funding, day_of_week,
        day_of_month, week_of_month, month_of_year, loans_in_week, loans_in_month,
        loans_on_day_of_week, loans_on_day_of_month, amount_in_week, amount_in_month,
        amount_on_day_of_week, amount_on_day_of_month, total_loans_in_month_across_years,
        total_amount_in_month_across_years, total_loans_in_year, total_amount_in_year
),

calendar_channels AS (
    SELECT
        c.Calendar_Date, ch.channel, c.Biz_Day, c.Funding_Day,
        c.Funding_Day_Remaining_in_Month, c.Biz_Day_Remaining_in_Month,
        c.Is_Holiday,
        DATEPART(WEEKDAY, c.Calendar_Date) - 1 AS day_of_week,
        DATEPART(DAY, c.Calendar_Date) AS day_of_month,
        DATEDIFF(WEEK, DATEADD(MONTH, DATEDIFF(MONTH, 0, c.Calendar_Date), 0), c.Calendar_Date) + 1 AS week_of_month,
        DATEPART(MONTH, c.Calendar_Date) AS month_of_year
    FROM marketing_sandbox.dbo.Calendar c
    CROSS JOIN channels ch
    WHERE c.Calendar_Date >= DATEADD(YEAR, -5, CAST(GETDATE() AS DATE))
      AND c.Calendar_Date <= (
            SELECT MIN(Calendar_Date)
            FROM (
                SELECT Calendar_Date,
                       ROW_NUMBER() OVER (ORDER BY Calendar_Date ASC) AS rn
                FROM marketing_sandbox.dbo.Calendar WITH (NOLOCK)
                WHERE Calendar_Date > CAST(GETDATE() AS DATE)
                  AND Biz_Day = 1
            ) biz
            WHERE rn = 3
      )
),

joined AS (
    SELECT
        cc.Calendar_Date, cc.channel, cc.Biz_Day, cc.Funding_Day,
        cc.Funding_Day_Remaining_in_Month, cc.Biz_Day_Remaining_in_Month,
        cc.Is_Holiday, cc.day_of_week, cc.day_of_month, cc.week_of_month,
        cc.month_of_year,
        ISNULL(al.number_of_loans, 0)            AS number_of_loans,
        ISNULL(al.total_loan_amount, 0)           AS total_loan_amount,
        ISNULL(al.week_weight, 0)                 AS week_weight,
        ISNULL(al.day_of_week_weight, 0)          AS day_of_week_weight,
        ISNULL(al.day_of_month_weight, 0)         AS day_of_month_weight,
        ISNULL(al.week_amount_weight, 0)          AS week_amount_weight,
        ISNULL(al.day_of_week_amount_weight, 0)   AS day_of_week_amount_weight,
        ISNULL(al.day_of_month_amount_weight, 0)  AS day_of_month_amount_weight,
        ISNULL(al.month_to_month_weight_loans, 0) AS month_to_month_weight_loans,
        ISNULL(al.month_to_month_weight_amount, 0)AS month_to_month_weight_amount,
        ISNULL(al.month_within_year_weight_loans, 0)  AS month_within_year_weight_loans,
        ISNULL(al.month_within_year_weight_amount, 0) AS month_within_year_weight_amount
    FROM calendar_channels cc
    LEFT JOIN agg_loans al
        ON cc.Calendar_Date = al.funded_date
       AND cc.channel = al.channel
)

SELECT
    Calendar_Date, channel, Biz_Day, Funding_Day,
    Funding_Day_Remaining_in_Month, Biz_Day_Remaining_in_Month,
    Is_Holiday, day_of_week, day_of_month, week_of_month, month_of_year,
    CASE WHEN day_of_week IN (5,6) THEN 0 ELSE number_of_loans END          AS number_of_loans,
    CASE WHEN day_of_week IN (5,6) THEN 0 ELSE total_loan_amount END         AS total_loan_amount,
    CASE WHEN day_of_week IN (5,6) THEN 0 ELSE week_weight END               AS week_weight,
    CASE WHEN day_of_week IN (5,6) THEN 0 ELSE day_of_week_weight END        AS day_of_week_weight,
    CASE WHEN day_of_week IN (5,6) THEN 0 ELSE day_of_month_weight END       AS day_of_month_weight,
    CASE WHEN day_of_week IN (5,6) THEN 0 ELSE week_amount_weight END        AS week_amount_weight,
    CASE WHEN day_of_week IN (5,6) THEN 0 ELSE day_of_week_amount_weight END AS day_of_week_amount_weight,
    CASE WHEN day_of_week IN (5,6) THEN 0 ELSE day_of_month_amount_weight END AS day_of_month_amount_weight,
    CASE WHEN day_of_week IN (5,6) THEN 0 ELSE month_to_month_weight_loans END  AS month_to_month_weight_loans,
    CASE WHEN day_of_week IN (5,6) THEN 0 ELSE month_to_month_weight_amount END AS month_to_month_weight_amount,
    CASE WHEN day_of_week IN (5,6) THEN 0 ELSE month_within_year_weight_loans END  AS month_within_year_weight_loans,
    CASE WHEN day_of_week IN (5,6) THEN 0 ELSE month_within_year_weight_amount END AS month_within_year_weight_amount
FROM joined
ORDER BY Calendar_Date ASC, channel;
"""

        conn    = pyodbc.connect(conn_str)
        cursor  = conn.cursor()
        cursor.execute(select_query)
        columns = [column[0] for column in cursor.description]
        data    = cursor.fetchall()
        df      = pd.DataFrame.from_records(data, columns=columns)
        print(f"Data loaded successfully. Shape: {df.shape}")
        cursor.close()
        conn.close()

    except Exception as e:
        print(f"An error occurred: {e}")
        if 'conn' in locals():
            conn.close()
else:
    print("Failed to retrieve secrets. Check your configuration.")

In [ ]:
#/////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////
#                                     Connect to SQL Server: Load in Historical Data                                                 #
#/////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////

def get_secret(secret_name, region_name):
    """
    Retrieve secrets from AWS Secrets Manager.
    """
    try:
        session = boto3.session.Session()
        client  = session.client(service_name="secretsmanager", region_name=region_name)
        response = client.get_secret_value(SecretId=secret_name)
        secret   = json.loads(response["SecretString"])
        return secret
    except Exception as e:
        print(f"Failed to retrieve secret: {e}")
        return None

secret_name = "analytics/prod/credentials"
region_name = "us-east-1"

secret = get_secret(secret_name, region_name)

if secret:
    try:
        server   = secret["datamart"]["host"]
        port     = secret["datamart"]["port"]
        database = secret["datamart"]["db"]
        username = secret["datamart"]["user"]
        password = secret["datamart"]["password"]

        conn_str = (
            f"Driver={{ODBC Driver 17 for SQL Server}};"
            f"Server={server},{port};"
            f"Database={database};"
            f"UID={username};"
            f"PWD={password};"
        )

        select_query = """
WITH app_counts AS (
    SELECT 
        dt,
        channel,
        COUNT(DISTINCT loan_number) AS application_count,
        SUM(loan_amount) AS application_volume
    FROM (
        SELECT DISTINCT
            CAST(b.unified_app_date AS DATE) AS dt,
            b.channel,
            b.loan_number,
            a.loan_amount
        FROM marketing_sandbox.dbo.SDS b WITH (NOLOCK)
        JOIN marketing_sandbox.dbo.skinny_core a
            ON a.loan_number = b.loan_number
        WHERE b.unified_app_date IS NOT NULL
    ) x
    GROUP BY dt, channel
),

uw_event_counts AS (
    SELECT 
        dt,
        channel,
        COUNT(DISTINCT loan_number) AS underwriting_submission_events,
        SUM(loan_amount) AS underwriting_submission_event_volume
    FROM (
        SELECT DISTINCT
            CAST(b.underwriting_submission_date AS DATE) AS dt,
            b.channel,
            b.loan_number,
            a.loan_amount
        FROM marketing_sandbox.dbo.SDS b WITH (NOLOCK)
        JOIN marketing_sandbox.dbo.skinny_core a
            ON a.loan_number = b.loan_number
        WHERE b.underwriting_submission_date IS NOT NULL
    ) x
    GROUP BY dt, channel
),

approval_event_counts AS (
    SELECT 
        dt,
        channel,
        COUNT(DISTINCT loan_number) AS approval_events,
        SUM(loan_amount) AS approval_event_volume
    FROM (
        SELECT DISTINCT
            CAST(b.initial_conditional_approval_date AS DATE) AS dt,
            b.channel,
            b.loan_number,
            a.loan_amount
        FROM marketing_sandbox.dbo.SDS b WITH (NOLOCK)
        JOIN marketing_sandbox.dbo.skinny_core a
            ON a.loan_number = b.loan_number
        WHERE b.initial_conditional_approval_date IS NOT NULL
    ) x
    GROUP BY dt, channel
),

base AS (
    SELECT
        CAST(a.filedt AS DATE) AS filedt,
        a.channel,

        COUNT(DISTINCT a.loan_number) AS loan_count,
        SUM(a.loan_amount) AS loan_volume,

        COUNT(DISTINCT CASE WHEN b.funded_date IS NOT NULL THEN a.loan_number END) AS funded_loan_count,
        SUM(CASE WHEN b.funded_date IS NOT NULL THEN a.loan_amount ELSE 0 END) AS funded_loan_volume,

        -- status-based counts
        COUNT(DISTINCT CASE WHEN b.underwriting_submission_date IS NOT NULL THEN a.loan_number END)
            AS underwriting_submission_count,

        COUNT(DISTINCT CASE WHEN b.initial_conditional_approval_date IS NOT NULL THEN a.loan_number END)
            AS initial_conditional_approval_count,

        -- event-based counts + volumes
        ISNULL(uw.underwriting_submission_events, 0) AS underwriting_submission_events,
        ISNULL(uw.underwriting_submission_event_volume, 0) AS underwriting_submission_event_volume,

        ISNULL(ap.approval_events, 0) AS approval_events,
        ISNULL(ap.approval_event_volume, 0) AS approval_event_volume,

        ISNULL(ac.application_count, 0) AS application_count,
        ISNULL(ac.application_volume, 0) AS application_volume,

        -- pull-through (count)
        CAST(
            100.0 * COUNT(DISTINCT CASE WHEN b.funded_date IS NOT NULL THEN a.loan_number END)
            / NULLIF(COUNT(DISTINCT a.loan_number), 0)
            AS DECIMAL(10,2)
        ) AS pull_through_pct_count,

        -- pull-through (volume)
        CAST(
            100.0 * SUM(CASE WHEN b.funded_date IS NOT NULL THEN a.loan_amount ELSE 0 END)
            / NULLIF(SUM(a.loan_amount), 0)
            AS DECIMAL(10,2)
        ) AS pull_through_pct_volume,

        -- =========================
        -- AVERAGE LOAN SIZE METRICS
        -- =========================

        CAST(
            ISNULL(ac.application_volume, 0) * 1.0
            / NULLIF(ac.application_count, 0)
            AS DECIMAL(18,2)
        ) AS avg_application_loan_size,

        CAST(
            ISNULL(uw.underwriting_submission_event_volume, 0) * 1.0
            / NULLIF(uw.underwriting_submission_events, 0)
            AS DECIMAL(18,2)
        ) AS avg_underwriting_submission_loan_size,

        CAST(
            ISNULL(ap.approval_event_volume, 0) * 1.0
            / NULLIF(ap.approval_events, 0)
            AS DECIMAL(18,2)
        ) AS avg_approval_loan_size

    FROM marketing_sandbox.dbo.skinny_core a WITH (NOLOCK)

    JOIN marketing_sandbox.dbo.SDS b WITH (NOLOCK)
        ON a.loan_number = b.loan_number

    LEFT JOIN app_counts ac
        ON CAST(a.filedt AS DATE) = ac.dt
        AND a.channel = ac.channel

    LEFT JOIN uw_event_counts uw
        ON CAST(a.filedt AS DATE) = uw.dt
        AND a.channel = uw.channel

    LEFT JOIN approval_event_counts ap
        ON CAST(a.filedt AS DATE) = ap.dt
        AND a.channel = ap.channel

    WHERE repeat_application = 0
      AND a.loan_status NOT LIKE '%C%'
      AND a.loan_status NOT IN (
            '0110','0115','0120','0125','0130','0140','0150','0160','0170','0180','0182',
            '0189','0190','0191','0192','0360','0362','0370','0375','0510','0610',
            '0618','0710','0711','0715','0720','1040'
      )

    GROUP BY 
        CAST(a.filedt AS DATE),
        a.channel,
        ac.application_count,
        ac.application_volume,
        uw.underwriting_submission_events,
        uw.underwriting_submission_event_volume,
        ap.approval_events,
        ap.approval_event_volume
)

SELECT
    b.*,

    c.Biz_Day,
    c.Biz_Day_Remaining_in_Month,
    c.Biz_Days_in_Month,
    c.Is_Holiday,
    c.Is_Company_Holiday,
    c.Calendar_Days_Remaining

FROM base b

LEFT JOIN marketing_sandbox.dbo.Calendar c
    ON b.filedt = CAST(c.Calendar_Date AS DATE)

WHERE c.Is_Weekday = 1

ORDER BY b.filedt;
"""

        conn    = pyodbc.connect(conn_str)
        cursor  = conn.cursor()
        cursor.execute(select_query)
        columns = [column[0] for column in cursor.description]
        data    = cursor.fetchall()
        pipeline_df      = pd.DataFrame.from_records(data, columns=columns)
        print(f"Data loaded successfully. Shape: {pipeline_df.shape}")
        cursor.close()
        conn.close()

    except Exception as e:
        print(f"An error occurred: {e}")
        if 'conn' in locals():
            conn.close()
else:
    print("Failed to retrieve secrets. Check your configuration.")

Data loaded successfully. Shape: (1596, 19)


In [11]:
#/////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////
#                                                          Data Preprocessing                                                        #
#/////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////

df['Calendar_Date'] = pd.to_datetime(df['Calendar_Date'])
df.fillna(0, inplace=True)

pipeline_df['filedt'] = pd.to_datetime(pipeline_df['filedt'])
pipeline_df.fillna(0, inplace=True)

In [12]:
#/////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////
#                                                          Split by Channel                                                          #
#/////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////

wholesale_historical    = df[df['channel'] == 'Wholesale'].copy()
retail_historical       = df[df['channel'] == 'Retail'].copy()
retail_broker_historical= df[df['channel'] == 'Retail Broker'].copy()
corr_historical         = df[df['channel'] == 'Correspondent'].copy()

pipeline_wholesale    = pipeline_df[pipeline_df['channel'] == 'Wholesale'].copy()
pipeline_retail       = pipeline_df[pipeline_df['channel'] == 'Retail'].copy()
pipeline_retail_broker = pipeline_df[pipeline_df['channel'] == 'Retail Broker'].copy()

In [13]:
#////////////////////////////////////////////////////////////////////////////////////////////////////////
#                                   MASTER HOLIDAY CALENDAR (2021–2029)
#////////////////////////////////////////////////////////////////////////////////////////////////////////

def build_us_holiday_calendar(start_date='2021-01-01', end_date='2029-12-31'):
    """
    Builds full daily calendar spine with hard holiday flags (Is_Holiday).
    """
    full_dates = pd.date_range(start=start_date, end=end_date, freq='D')
    calendar   = pd.DataFrame({'Calendar_Date': full_dates})
    calendar['Calendar_Date'] = pd.to_datetime(calendar['Calendar_Date'])

    cal      = USFederalHolidayCalendar()
    holidays = cal.holidays(start=start_date, end=end_date)

    calendar['Is_Holiday'] = calendar['Calendar_Date'].isin(holidays).astype(int)
    return calendar[['Calendar_Date', 'Is_Holiday']]

In [14]:
#////////////////////////////////////////////////////////////////////////////////////////////////////////
#                          HolidayDistortionLearner
#  Learns per-(holiday_dow, business_day_offset) distortion factors from history.
#  e.g. "The Tuesday after a Monday holiday historically funds +35% above normal"
#////////////////////////////////////////////////////////////////////////////////////////////////////////

class HolidayDistortionLearner:
    """
    Learns empirical holiday distortion factors from historical data.

    For each holiday in training history:
      - Identifies the holiday's day-of-week (holiday_dow)
      - Looks at surrounding business days in a window [-2, +4]
      - Computes ratio of actual / model-expected for each offset day
      - Aggregates across years using median + shrinkage toward 1.0

    Result: distortion_table keyed by (holiday_dow, biz_offset)
    """

    def __init__(self,
                 window_before=2,
                 window_after=4,
                 shrinkage=0.3,
                 min_observations=2):

        self.window_before = window_before
        self.window_after = window_after
        self.shrinkage = shrinkage
        self.min_observations = min_observations
        self.distortion_table_ = {}

    def fit(self, df, holiday_calendar, expected_col='expected'):
        """
        df            : training dataframe with Calendar_Date, actual target, and expected column
        holiday_calendar : dataframe with Calendar_Date and Is_Holiday columns
        expected_col  : column name in df containing the model's baseline expected value
        """

        df = df.copy()
        df['Calendar_Date'] = pd.to_datetime(df['Calendar_Date'])

        holiday_calendar = holiday_calendar.copy()
        holiday_calendar['Calendar_Date'] = pd.to_datetime(
            holiday_calendar['Calendar_Date']
        )

        holiday_dates = holiday_calendar.loc[
            holiday_calendar['Is_Holiday'] == 1, 'Calendar_Date'
        ].tolist()

        # Business day spine from training data (weekdays only, non-holiday)
        biz_days = df.loc[
            (~df['Calendar_Date'].dt.dayofweek.isin([5, 6]))
        ].copy()

        biz_days = biz_days.sort_values('Calendar_Date').reset_index(drop=True)

        # Map date -> position in business day sequence
        biz_day_index = {
            row['Calendar_Date']: idx
            for idx, row in biz_days.iterrows()
        }

        # Accumulate ratios: {(holiday_dow, offset): [ratio, ratio, ...]}
        ratio_accumulator = {}

        for hdate in holiday_dates:

            holiday_dow = hdate.dayofweek

            # Skip weekends (shouldn't happen with USFederalHolidayCalendar observed)
            if holiday_dow in [5, 6]:
                continue

            # Find surrounding business days
            for offset in range(-self.window_before, self.window_after + 1):

                if offset == 0:
                    continue  # Holiday itself is always hard zero

                # Step through calendar days to find the Nth business day offset
                candidate = hdate
                steps = 0
                direction = 1 if offset > 0 else -1
                abs_offset = abs(offset)

                while steps < abs_offset:
                    candidate = candidate + pd.Timedelta(days=direction)
                    if candidate.dayofweek not in [5, 6]:
                        steps += 1

                # Look up actual and expected for this candidate date
                row = df.loc[df['Calendar_Date'] == candidate]

                if row.empty:
                    continue

                actual = row[expected_col.replace('expected', df.columns[
                    [c for c in df.columns if 'loan_amount' in c or 'loans' in c][0]
                    if any('loan_amount' in c or 'loans' in c for c in df.columns)
                    else -1
                ])].values[0] if False else None

                # Simpler: use whatever target column is available
                target_cols = [c for c in df.columns
                               if 'total_loan_amount' in c or 'number_of_loans' in c]

                if not target_cols:
                    continue

                target_col = target_cols[0]
                actual = row[target_col].values[0]
                expected = row[expected_col].values[0]

                if expected <= 0 or actual < 0:
                    continue

                ratio = actual / expected

                key = (holiday_dow, offset)

                if key not in ratio_accumulator:
                    ratio_accumulator[key] = []

                ratio_accumulator[key].append(ratio)

        # Aggregate: median + shrinkage toward 1.0
        self.distortion_table_ = {}

        for key, ratios in ratio_accumulator.items():

            if len(ratios) < self.min_observations:
                continue

            median_ratio = np.median(ratios)

            shrunk = (
                (1 - self.shrinkage) * median_ratio +
                self.shrinkage * 1.0
            )

            self.distortion_table_[key] = shrunk

        return self

    def get_distortion(self, holiday_dow, biz_offset):
        """
        Returns learned distortion factor for a given
        (holiday day-of-week, business day offset from holiday).
        Defaults to 1.0 if no data learned.
        """
        return self.distortion_table_.get((holiday_dow, biz_offset), 1.0)

In [15]:
#///////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////
#                                         Model A — StructuralLoanForecaster_A                                                     #
#///////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////

class StructuralLoanForecaster_A:

    def __init__(self,
                 target='total_loan_amount',
                 dow_shrinkage=0.25,
                 interaction_shrinkage=0.35,
                 seasonal_exponent=1.02,
                 recency_strength=0.35,
                 trend_seasonal_tilt=0.08,
                 level_lookback_days=60,
                 trend_dampening=0.83,
                 forward_lift_daily=0.0025,
                 holiday_window_before=2,
                 holiday_window_after=4,
                 holiday_shrinkage=0.3):

        self.target = target
        self.dow_shrinkage = dow_shrinkage
        self.interaction_shrinkage = interaction_shrinkage
        self.seasonal_exponent = seasonal_exponent
        self.recency_strength = recency_strength
        self.trend_seasonal_tilt = trend_seasonal_tilt
        self.level_lookback_days = level_lookback_days
        self.trend_dampening = trend_dampening
        self.forward_lift_daily = forward_lift_daily
        self.holiday_window_before = holiday_window_before
        self.holiday_window_after = holiday_window_after
        self.holiday_shrinkage = holiday_shrinkage

        self.master_calendar = None
        self.distortion_table_ = {}
        self.fitted = False

    # ---------------------------------------------------
    # INTERNAL: compute expected baseline for a dataframe
    # ---------------------------------------------------
    def _compute_expected(self, df, t_array):
        """
        Given a df with dow/dom/moy columns and a t_array of
        trend indices, returns the baseline expected value array
        (trend × DOW × DOM × MOY × interaction).
        """

        trend = np.exp(
            self.trend_intercept + self.trend_slope * t_array
        )

        dow_e = df['dow'].map(self.dow_effect).fillna(1.0).values
        dom_e = df['dom'].map(self.dom_effect).fillna(1.0).values
        moy_e = df['moy'].map(self.moy_effect).fillna(1.0).values

        interaction_e = np.array([
            self.interaction_effect.get((d, m), 1.0)
            for d, m in zip(df['dow'], df['moy'])
        ])

        seasonal = dow_e * dom_e * moy_e * interaction_e
        seasonal = seasonal / np.mean(seasonal)
        seasonal = seasonal ** self.seasonal_exponent

        tilt = 1 + self.trend_seasonal_tilt * (seasonal - 1)

        return self.base_level * seasonal * trend * tilt

    # ---------------------------------------------------
    # FIT
    # ---------------------------------------------------
    def fit(self, df, holiday_calendar=None):

        df = df.copy()
        df = df.sort_values('Calendar_Date')
        df['Calendar_Date'] = pd.to_datetime(df['Calendar_Date'])

        df['dow'] = df['Calendar_Date'].dt.dayofweek
        df['dom'] = df['Calendar_Date'].dt.day
        df['moy'] = df['Calendar_Date'].dt.month
        df['is_weekend'] = df['dow'].isin([5, 6])

        # ---- Store master holiday calendar ----
        if holiday_calendar is not None:
            calendar = holiday_calendar.copy()
            calendar['Calendar_Date'] = pd.to_datetime(calendar['Calendar_Date'])
            self.master_calendar = calendar

            df = df.merge(
                calendar[['Calendar_Date', 'Is_Holiday']],
                on='Calendar_Date',
                how='left'
            )
            df['Is_Holiday'] = df['Is_Holiday'].fillna(0)

        else:
            df['Is_Holiday'] = 0
            self.master_calendar = None

        # ---- Production mask: weekdays, non-holiday, non-zero ----
        prod_mask = (
            (~df['is_weekend']) &
            (df['Is_Holiday'] == 0) &
            (df[self.target] > 0)
        )

        # ---- NEW: Seasonal mask — full completed months only ----
        last_date           = df['Calendar_Date'].max()
        partial_month_start = last_date.replace(day=1)

        seasonal_mask = (
            prod_mask &
            (df['Calendar_Date'] < partial_month_start)
        )

        df_prod = df.loc[prod_mask].copy()

        self.last_train_date = df['Calendar_Date'].max()
        self.n_train = len(df)

        # ---- Trend (log-linear WLS, recency weighted) — FAST VERSION ----
        t_prod = np.arange(len(df_prod))
        recency_prod = np.exp(self.recency_strength * (t_prod / len(df_prod)))

        y = np.log(df_prod[self.target].values)
        X = np.vstack([np.ones(len(t_prod)), t_prod]).T

        sqrt_w = np.sqrt(recency_prod)
        Xw     = X * sqrt_w[:, None]
        yw     = y * sqrt_w
        beta   = np.linalg.lstsq(Xw, yw, rcond=None)[0]

        self.trend_intercept = beta[0]
        self.trend_slope = beta[1]

        t_full = np.arange(len(df))
        trend_fitted = np.exp(self.trend_intercept + self.trend_slope * t_full)

        df['trend_fitted'] = trend_fitted
        df['detrended'] = df[self.target] / trend_fitted

        # ---- Base level — uses prod_mask (includes partial month) ----
        recent_prod = df.loc[prod_mask].tail(self.level_lookback_days)
        self.base_level = (
            recent_prod[self.target].mean() /
            recent_prod['trend_fitted'].mean()
        )

        # ---- DOW effect — uses seasonal_mask (full months only) ----
        dow_raw = df.loc[seasonal_mask].groupby('dow')['detrended'].mean()
        dow_shrunk = (
            (1 - self.dow_shrinkage) * dow_raw +
            self.dow_shrinkage * self.base_level
        )
        self.dow_effect = dow_shrunk / dow_shrunk.mean()
        self.dow_effect.loc[5] = 0.0
        self.dow_effect.loc[6] = 0.0

        df['after_dow'] = df['detrended'] / df['dow'].map(
            lambda x: self.dow_effect.get(x, 1.0)
        )

        # ---- DOM effect — uses seasonal_mask ----
        dom_raw = df.loc[seasonal_mask].groupby('dom')['after_dow'].mean()
        self.dom_effect = dom_raw / dom_raw.mean()
        df['after_dom'] = df['after_dow'] / df['dom'].map(self.dom_effect)

        # ---- MOY effect — uses seasonal_mask ----
        moy_raw = df.loc[seasonal_mask].groupby('moy')['after_dom'].mean()
        self.moy_effect = moy_raw / moy_raw.mean()
        df['after_moy'] = df['after_dom'] / df['moy'].map(self.moy_effect)

        # ---- DOW x MOY interaction — uses seasonal_mask ----
        interaction_raw = (
            df.loc[seasonal_mask]
            .groupby(['dow', 'moy'])['after_moy']
            .mean()
        )
        interaction_shrunk = (
            (1 - self.interaction_shrinkage) * interaction_raw +
            self.interaction_shrinkage * interaction_raw.mean()
        )
        self.interaction_effect = (
            interaction_shrunk / interaction_shrunk.mean()
        )

        # ================================================
        # LEARN HOLIDAY DISTORTION FACTORS — uses prod_mask
        # ================================================
        if self.master_calendar is not None:

            t_full_arr = np.arange(len(df))
            df['expected'] = self._compute_expected(df, t_full_arr)

            holiday_dates = self.master_calendar.loc[
                self.master_calendar['Is_Holiday'] == 1, 'Calendar_Date'
            ].tolist()

            biz_day_list = df.loc[
                ~df['is_weekend']
            ]['Calendar_Date'].sort_values().tolist()

            biz_day_pos = {d: i for i, d in enumerate(biz_day_list)}

            ratio_accumulator = {}

            for hdate in holiday_dates:

                holiday_dow = hdate.dayofweek

                if holiday_dow in [5, 6]:
                    continue

                if hdate not in biz_day_pos:
                    continue

                h_pos = biz_day_pos[hdate]

                for offset in range(
                    -self.holiday_window_before,
                    self.holiday_window_after + 1
                ):

                    if offset == 0:
                        continue

                    target_pos = h_pos + offset

                    if target_pos < 0 or target_pos >= len(biz_day_list):
                        continue

                    target_date = biz_day_list[target_pos]

                    row = df.loc[df['Calendar_Date'] == target_date]

                    if row.empty:
                        continue

                    if row['Is_Holiday'].values[0] == 1:
                        continue
                    if row['is_weekend'].values[0]:
                        continue

                    actual = row[self.target].values[0]
                    expected = row['expected'].values[0]

                    if expected <= 0 or actual <= 0:
                        continue

                    ratio = actual / expected
                    key = (holiday_dow, offset)

                    if key not in ratio_accumulator:
                        ratio_accumulator[key] = []

                    ratio_accumulator[key].append(ratio)

            self.distortion_table_ = {}

            for key, ratios in ratio_accumulator.items():

                if len(ratios) < 2:
                    continue

                median_ratio = float(np.median(ratios))

                shrunk = (
                    (1 - self.holiday_shrinkage) * median_ratio +
                    self.holiday_shrinkage * 1.0
                )

                self.distortion_table_[key] = shrunk

        self.fitted = True
        return self

    # ---------------------------------------------------
    # FORECAST
    # ---------------------------------------------------
    def forecast_from_date(self, start_date, months_forward=1):

        if not self.fitted:
            raise Exception('Model must be fit first.')

        start_date = pd.to_datetime(start_date)
        end_date = start_date + pd.DateOffset(months=months_forward)

        future_dates = pd.date_range(
            start=start_date,
            end=end_date - pd.Timedelta(days=1),
            freq='D'
        )

        df = pd.DataFrame({'Calendar_Date': future_dates})
        df['dow'] = df['Calendar_Date'].dt.dayofweek
        df['dom'] = df['Calendar_Date'].dt.day
        df['moy'] = df['Calendar_Date'].dt.month
        df['is_weekend'] = df['dow'].isin([5, 6])

        # ---- Trend ----
        t_future = np.arange(self.n_train, self.n_train + len(df))

        trend_multiplier = np.exp(
            self.trend_intercept +
            (self.trend_slope * self.trend_dampening) * t_future
        )

        horizon_index = np.arange(len(df))
        forward_lift = 1 + self.forward_lift_daily * horizon_index

        # ---- Seasonal ----
        dow_e = df['dow'].map(self.dow_effect).fillna(1.0).values
        dom_e = df['dom'].map(self.dom_effect).fillna(1.0).values
        moy_e = df['moy'].map(self.moy_effect).fillna(1.0).values

        interaction_e = np.array([
            self.interaction_effect.get((d, m), 1.0)
            for d, m in zip(df['dow'], df['moy'])
        ])

        seasonal = dow_e * dom_e * moy_e * interaction_e
        seasonal = seasonal / np.mean(seasonal)
        seasonal = seasonal ** self.seasonal_exponent

        tilt = 1 + self.trend_seasonal_tilt * (seasonal - 1)

        forecast = (
            self.base_level
            * seasonal
            * trend_multiplier
            * tilt
            * forward_lift
        )

        # ---- Weekend hard zero ----
        forecast[df['is_weekend'].values] = 0.0

        # ---- Apply holiday calendar ----
        if self.master_calendar is not None:

            df = df.merge(
                self.master_calendar,
                on='Calendar_Date',
                how='left'
            )
            df['Is_Holiday'] = df['Is_Holiday'].fillna(0)

            # Hard zero on holiday itself
            forecast[df['Is_Holiday'].values == 1] = 0.0

            holiday_dates_forecast = self.master_calendar.loc[
                (self.master_calendar['Is_Holiday'] == 1) &
                (self.master_calendar['Calendar_Date'] >= start_date) &
                (self.master_calendar['Calendar_Date'] <= future_dates[-1])
            ]['Calendar_Date'].tolist()

            lookback_start = start_date - pd.Timedelta(
                days=(self.holiday_window_before + 1) * 2
            )

            holiday_dates_nearby = self.master_calendar.loc[
                (self.master_calendar['Is_Holiday'] == 1) &
                (self.master_calendar['Calendar_Date'] >= lookback_start) &
                (self.master_calendar['Calendar_Date'] < start_date)
            ]['Calendar_Date'].tolist()

            all_relevant_holidays = holiday_dates_nearby + holiday_dates_forecast

            forecast_biz_days = df.loc[
                ~df['is_weekend']
            ]['Calendar_Date'].sort_values().tolist()

            forecast_biz_pos = {d: i for i, d in enumerate(forecast_biz_days)}

            distortion = np.ones(len(df))

            for hdate in all_relevant_holidays:

                holiday_dow = hdate.dayofweek

                if holiday_dow in [5, 6]:
                    continue

                for offset in range(
                    -self.holiday_window_before,
                    self.holiday_window_after + 1
                ):

                    if offset == 0:
                        continue

                    candidate = hdate
                    steps = 0
                    direction = 1 if offset > 0 else -1
                    abs_offset = abs(offset)

                    while steps < abs_offset:
                        candidate = candidate + pd.Timedelta(days=direction)
                        if candidate.dayofweek not in [5, 6]:
                            steps += 1

                    row_idx = df.index[df['Calendar_Date'] == candidate].tolist()

                    if not row_idx:
                        continue

                    idx = row_idx[0]

                    factor = self.distortion_table_.get(
                        (holiday_dow, offset), 1.0
                    )

                    distortion[idx] = factor

            forecast = forecast * distortion

        df['forecast'] = forecast

        return df

In [16]:
#///////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////
#                                         Model B — StructuralLoanForecaster_B                                                     #
#///////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////

class StructuralLoanForecaster_B:
    """
    Structural time-series forecaster for daily loan volumes.

    Production parameters (validated Aug 2024 – Nov 2025 backtest):
        trend_dampening          = 0.800
        distortion_max_factor    = 1.2
        distortion_lookback_days = 730
        distortion_min_obs       = 3

    Backtest results (16 months):
        MAE        5.51%
        Median AE  4.16%
        Within 5%  8/16
        Within 10% 13/16
        Max error  +20.1% (Jan 2025 — 2025 H1 hyper-growth, not modelable)
    """

    def __init__(
        self,
        target                    = 'total_loan_amount',
        recency_strength          = 1.0,
        dow_shrinkage             = 0.05,
        interaction_shrinkage     = 0.2,
        level_lookback_days       = 60,
        holiday_window_before     = 2,
        holiday_window_after      = 4,
        holiday_shrinkage         = 0.2,
        distortion_min_obs        = 3,
        distortion_max_factor     = 1.1,
        distortion_lookback_days  = 730,
        per_key_caps              = None,
        mon_thu_override_fraction = None,
        trend_dampening           = 0.70,
        forward_lift_daily        = 0.0,
        trend_seasonal_tilt       = 0.0,
        seasonal_exponent         = 1.0,
        moy_recency_strength      = 0.0,
        mom_dampening             = 0.0,
    ):
        self.target                    = target
        self.recency_strength          = recency_strength
        self.dow_shrinkage             = dow_shrinkage
        self.interaction_shrinkage     = interaction_shrinkage
        self.level_lookback_days       = level_lookback_days
        self.holiday_window_before     = holiday_window_before
        self.holiday_window_after      = holiday_window_after
        self.holiday_shrinkage         = holiday_shrinkage
        self.distortion_min_obs        = distortion_min_obs
        self.distortion_max_factor     = distortion_max_factor
        self.distortion_lookback_days  = distortion_lookback_days
        self.per_key_caps              = per_key_caps or {}
        self.mon_thu_override_fraction = mon_thu_override_fraction
        self.trend_dampening           = trend_dampening
        self.forward_lift_daily        = forward_lift_daily
        self.trend_seasonal_tilt       = trend_seasonal_tilt
        self.seasonal_exponent         = seasonal_exponent
        self.moy_recency_strength      = moy_recency_strength
        self.mom_dampening             = mom_dampening
        self.fitted                    = False

    # ------------------------------------------------------------------
    def fit(self, df, holiday_calendar=None):

        df = df.copy()
        df = df.sort_values('Calendar_Date').reset_index(drop=True)
        df['Calendar_Date'] = pd.to_datetime(df['Calendar_Date'])
        df['dow']        = df['Calendar_Date'].dt.dayofweek
        df['dom']        = df['Calendar_Date'].dt.day
        df['moy']        = df['Calendar_Date'].dt.month
        df['is_weekend'] = df['dow'].isin([5, 6])

        if holiday_calendar is not None:
            calendar = holiday_calendar.copy()
            calendar['Calendar_Date'] = pd.to_datetime(calendar['Calendar_Date'])
            self.master_calendar = calendar
            df = df.merge(
                calendar[['Calendar_Date', 'Is_Holiday']],
                on='Calendar_Date', how='left'
            )
            df['Is_Holiday'] = df['Is_Holiday'].fillna(0)
        else:
            df['Is_Holiday'] = 0
            self.master_calendar = None

        df = df.reset_index(drop=True)

        prod_mask = (
            (~df['is_weekend']) &
            (df['Is_Holiday'] == 0) &
            (df[self.target] > 0)
        )

        # ---- Holiday halo (existing Model B logic) ----
        if self.master_calendar is not None:
            holiday_dates = set(pd.to_datetime(
                self.master_calendar.loc[
                    self.master_calendar['Is_Holiday'] == 1, 'Calendar_Date'
                ]
            ))
            halo_dates = set()
            for hdate in holiday_dates:
                for offset in range(
                    -self.holiday_window_before,
                     self.holiday_window_after + 1
                ):
                    candidate = hdate + pd.Timedelta(days=offset)
                    if candidate.dayofweek not in [5, 6]:
                        halo_dates.add(candidate)
            df['in_halo'] = df['Calendar_Date'].isin(halo_dates)
        else:
            holiday_dates = set()
            df['in_halo']  = False

        # ---- NEW: Seasonal mask — full completed months only + halo exclusion ----
        last_date           = df['Calendar_Date'].max()
        partial_month_start = last_date.replace(day=1)

        seasonal_mask = (
            prod_mask &
            (~df['in_halo']) &
            (df['Calendar_Date'] < partial_month_start)
        )

        df_prod = df.loc[prod_mask].copy()

        self.last_train_date = df['Calendar_Date'].max()
        self.n_train         = len(df)

        # ---- Trend — uses prod_mask (includes partial month) ----
        t_prod       = np.arange(len(df_prod))
        recency_prod = np.exp(self.recency_strength * (t_prod / len(df_prod)))
        y_log        = np.log(df_prod[self.target].values)
        X            = np.vstack([np.ones(len(t_prod)), t_prod]).T
        W            = np.diag(recency_prod)
        beta         = np.linalg.inv(X.T @ W @ X) @ (X.T @ W @ y_log)
        self.trend_intercept = beta[0]
        self.trend_slope     = beta[1]

        t_full       = np.arange(len(df))
        trend_fitted = np.exp(self.trend_intercept + self.trend_slope * t_full)

        df['trend_fitted'] = trend_fitted
        df['detrended']    = np.where(
            trend_fitted > 0,
            df[self.target] / trend_fitted,
            0.0
        )

        # ---- Base level — uses prod_mask (includes partial month) ----
        recent_prod     = df.loc[prod_mask].tail(self.level_lookback_days)
        target_mean     = float(recent_prod[self.target].mean())
        trend_mean      = float(recent_prod['trend_fitted'].mean())
        self.base_level = target_mean / trend_mean if trend_mean > 0 else 1.0

        # ---- DOW effect — uses seasonal_mask (full months, no halo) ----
        dow_raw    = df.loc[seasonal_mask].groupby('dow')['detrended'].mean()
        dow_shrunk = (
            (1 - self.dow_shrinkage) * dow_raw +
            self.dow_shrinkage * self.base_level
        )
        self.dow_effect        = dow_shrunk / dow_shrunk.mean()
        self.dow_effect.loc[5] = 0.0
        self.dow_effect.loc[6] = 0.0

        df['after_dow'] = df['detrended'] / df['dow'].map(
            lambda x: self.dow_effect.get(x, 1.0)
        )

        # ---- DOM effect — uses seasonal_mask ----
        dom_raw         = df.loc[seasonal_mask].groupby('dom')['after_dow'].mean()
        self.dom_effect = dom_raw / dom_raw.mean()
        df['after_dom'] = df['after_dow'] / df['dom'].map(self.dom_effect)

        # ---- MOY effect — uses seasonal_mask ----
        df_seas_moy = df.loc[seasonal_mask].copy()
        min_year    = df_seas_moy['Calendar_Date'].dt.year.min()
        max_year    = df_seas_moy['Calendar_Date'].dt.year.max()
        df_seas_moy['year']        = df_seas_moy['Calendar_Date'].dt.year
        df_seas_moy['year_weight'] = np.exp(
            self.moy_recency_strength *
            (df_seas_moy['year'] - min_year) / max(max_year - min_year, 1)
        )
        moy_raw = (
            df_seas_moy
            .groupby('moy')
            .apply(
                lambda g: np.average(g['after_dom'], weights=g['year_weight']),
                include_groups=False
            )
        )
        self.moy_effect = moy_raw / moy_raw.mean()
        df['after_moy'] = df['after_dom'] / df['moy'].map(self.moy_effect)

        # ---- DOW x MOY interaction — uses seasonal_mask ----
        interaction_raw = (
            df.loc[seasonal_mask]
            .groupby(['dow', 'moy'])['after_moy']
            .mean()
        )
        interaction_shrunk = (
            (1 - self.interaction_shrinkage) * interaction_raw +
            self.interaction_shrinkage * interaction_raw.mean()
        )
        self.interaction_effect = interaction_shrunk / interaction_shrunk.mean()

        # ---- Monthly totals — uses full df (unchanged) ----
        self.monthly_totals_ = (
            df.groupby([
                df['Calendar_Date'].dt.year,
                df['Calendar_Date'].dt.month
            ])[self.target].sum()
        )
        self.monthly_totals_.index.names = ['year', 'month']

        # ---- Distortion table — uses prod_mask (unchanged) ----
        if self.master_calendar is not None:

            last_train = df['Calendar_Date'].max()
            if self.distortion_lookback_days is not None:
                distortion_cutoff = last_train - pd.Timedelta(
                    days=self.distortion_lookback_days
                )
                df_dist = df[df['Calendar_Date'] >= distortion_cutoff].copy()
            else:
                df_dist = df.copy()

            df_dist = df_dist.reset_index(drop=True)
            t_dist  = np.arange(len(df_dist))
            df_dist['expected'] = self._compute_expected(df_dist, t_dist)

            biz_day_list = df_dist.loc[
                ~df_dist['is_weekend']
            ]['Calendar_Date'].sort_values().tolist()
            biz_day_pos  = {d: i for i, d in enumerate(biz_day_list)}

            ratio_accumulator = {}
            for hdate in holiday_dates:
                holiday_dow = hdate.dayofweek
                if holiday_dow in [5, 6]:
                    continue
                if hdate not in biz_day_pos:
                    continue
                h_pos = biz_day_pos[hdate]

                for offset in range(
                    -self.holiday_window_before,
                     self.holiday_window_after + 1
                ):
                    if offset == 0:
                        continue
                    if (
                        holiday_dow == 0 and offset == 3 and
                        self.mon_thu_override_fraction is not None
                    ):
                        continue
                    tp = h_pos + offset
                    if tp < 0 or tp >= len(biz_day_list):
                        continue
                    target_date = biz_day_list[tp]
                    row         = df_dist.loc[df_dist['Calendar_Date'] == target_date]
                    if row.empty:
                        continue
                    if row['Is_Holiday'].values[0] == 1:
                        continue
                    if row['is_weekend'].values[0]:
                        continue
                    actual   = row[self.target].values[0]
                    expected = row['expected'].values[0]
                    if expected <= 0 or actual <= 0:
                        continue
                    ratio = actual / expected
                    ratio_accumulator.setdefault((holiday_dow, offset), []).append(ratio)

            self.distortion_table_ = {}
            for key, ratios in ratio_accumulator.items():
                if len(ratios) < self.distortion_min_obs:
                    continue
                raw_median = float(np.median(ratios))
                shrunk     = (
                    (1 - self.holiday_shrinkage) * raw_median +
                    self.holiday_shrinkage * 1.0
                )
                if key in self.per_key_caps:
                    effective_cap = max(
                        self.distortion_max_factor,
                        self.per_key_caps[key]
                    )
                else:
                    effective_cap = self.distortion_max_factor

                capped = float(np.clip(
                    shrunk,
                    1.0 / effective_cap,
                    effective_cap
                ))
                self.distortion_table_[key] = capped
        else:
            self.distortion_table_ = {}

        self.fitted = True
        return self

    # ------------------------------------------------------------------
    def forecast_from_date(self, start_date, months_forward=1):

        if not self.fitted:
            raise Exception('Model must be fit before forecasting.')

        start_date = pd.to_datetime(start_date)
        end_date   = start_date + pd.DateOffset(months=months_forward)

        future_dates = pd.date_range(
            start=start_date,
            end=end_date - pd.Timedelta(days=1),
            freq='D'
        )

        df = pd.DataFrame({'Calendar_Date': future_dates})
        df['dow']        = df['Calendar_Date'].dt.dayofweek
        df['dom']        = df['Calendar_Date'].dt.day
        df['moy']        = df['Calendar_Date'].dt.month
        df['is_weekend'] = df['dow'].isin([5, 6])

        t_future         = np.arange(self.n_train, self.n_train + len(df))
        trend_multiplier = np.exp(
            self.trend_intercept +
            (self.trend_slope * self.trend_dampening) * t_future
        )

        horizon_index = np.arange(len(df))
        forward_lift  = 1 + self.forward_lift_daily * horizon_index

        dow_e = df['dow'].map(self.dow_effect).fillna(1.0).values
        dom_e = df['dom'].map(self.dom_effect).fillna(1.0).values
        moy_e = df['moy'].map(self.moy_effect).fillna(1.0).values
        interaction_e = np.array([
            self.interaction_effect.get((d, m), 1.0)
            for d, m in zip(df['dow'], df['moy'])
        ])

        seasonal = dow_e * dom_e * moy_e * interaction_e
        seasonal = seasonal / np.mean(seasonal)
        seasonal = seasonal ** self.seasonal_exponent
        tilt     = 1 + self.trend_seasonal_tilt * (seasonal - 1)

        forecast = (
            self.base_level * seasonal * trend_multiplier * tilt * forward_lift
        )

        for period in df['Calendar_Date'].dt.to_period('M').unique():
            yr, mo     = period.year, period.month
            weight     = self._mom_trend_weight(yr, mo)
            month_mask = (
                (df['Calendar_Date'].dt.year  == yr) &
                (df['Calendar_Date'].dt.month == mo)
            ).values
            forecast[month_mask] *= weight

        forecast[df['is_weekend'].values] = 0.0

        if self.master_calendar is not None:
            df = df.merge(self.master_calendar, on='Calendar_Date', how='left')
            df['Is_Holiday'] = df['Is_Holiday'].fillna(0)
            forecast[df['Is_Holiday'].values == 1] = 0.0

            holiday_dates_forecast = self.master_calendar.loc[
                (self.master_calendar['Is_Holiday'] == 1) &
                (self.master_calendar['Calendar_Date'] >= start_date) &
                (self.master_calendar['Calendar_Date'] <= future_dates[-1])
            ]['Calendar_Date'].tolist()

            lookback_start = start_date - pd.Timedelta(
                days=(self.holiday_window_before + 1) * 2
            )
            holiday_dates_nearby = self.master_calendar.loc[
                (self.master_calendar['Is_Holiday'] == 1) &
                (self.master_calendar['Calendar_Date'] >= lookback_start) &
                (self.master_calendar['Calendar_Date'] < start_date)
            ]['Calendar_Date'].tolist()

            all_relevant_holidays = holiday_dates_nearby + holiday_dates_forecast
            distortion            = np.ones(len(df))

            all_cal      = pd.date_range(start=lookback_start, end=future_dates[-1], freq='D')
            biz_days_ext = [d for d in all_cal if d.dayofweek not in [5, 6]]
            biz_pos_ext  = {d: i for i, d in enumerate(biz_days_ext)}

            for hdate in all_relevant_holidays:
                holiday_dow = hdate.dayofweek
                if holiday_dow in [5, 6]:
                    continue

                for offset in range(
                    -self.holiday_window_before,
                     self.holiday_window_after + 1
                ):
                    if offset == 0:
                        continue
                    if (
                        holiday_dow == 0 and offset == 3 and
                        self.mon_thu_override_fraction is not None
                    ):
                        continue

                    candidate  = hdate
                    steps      = 0
                    direction  = 1 if offset > 0 else -1
                    abs_offset = abs(offset)
                    while steps < abs_offset:
                        candidate = candidate + pd.Timedelta(days=direction)
                        if candidate.dayofweek not in [5, 6]:
                            steps += 1

                    row_idx = df.index[df['Calendar_Date'] == candidate].tolist()
                    if not row_idx:
                        continue
                    idx    = row_idx[0]
                    factor = self._get_distortion(holiday_dow, offset)
                    distortion[idx] = factor

            forecast = forecast * distortion

            if self.mon_thu_override_fraction is not None:
                for hdate in all_relevant_holidays:
                    if hdate.dayofweek != 0:
                        continue
                    if hdate not in biz_pos_ext:
                        continue
                    h_pos      = biz_pos_ext[hdate]
                    target_pos = h_pos + 3
                    if target_pos >= len(biz_days_ext):
                        continue
                    target_date = biz_days_ext[target_pos]
                    if target_date.dayofweek != 3:
                        continue
                    row_idx = df.index[df['Calendar_Date'] == target_date].tolist()
                    if not row_idx:
                        continue
                    idx   = row_idx[0]
                    t_pos = self.n_train + idx
                    trend_v   = np.exp(
                        self.trend_intercept +
                        (self.trend_slope * self.trend_dampening) * t_pos
                    )
                    moy       = target_date.month
                    dom_e_val = self.dom_effect.get(target_date.day, 1.0)
                    moy_e_val = self.moy_effect.get(moy, 1.0)
                    mon_dow_e = self.dow_effect.get(0, 1.0)
                    mon_int_e = self.interaction_effect.get((0, moy), 1.0)
                    expected_mon = (
                        self.base_level * trend_v *
                        dom_e_val * moy_e_val * mon_dow_e * mon_int_e
                    )
                    forecast[idx] = self.mon_thu_override_fraction * expected_mon

        df['forecast'] = forecast
        return df

    # ------------------------------------------------------------------
    def _compute_expected(self, df, t_arr):
        trend = np.exp(self.trend_intercept + self.trend_slope * t_arr)
        dow_e = df['dow'].map(self.dow_effect).fillna(1.0).values
        dom_e = df['dom'].map(self.dom_effect).fillna(1.0).values
        moy_e = df['moy'].map(self.moy_effect).fillna(1.0).values
        interaction_e = np.array([
            self.interaction_effect.get((d, m), 1.0)
            for d, m in zip(df['dow'], df['moy'])
        ])
        return self.base_level * trend * dow_e * dom_e * moy_e * interaction_e

    def _get_distortion(self, holiday_dow, offset):
        return self.distortion_table_.get((holiday_dow, offset), 1.0)

    def _mom_trend_weight(self, year, month):
        if self.mom_dampening == 0.0:
            return 1.0
        try:
            prev_year  = year - 1 if month > 1 else year - 2
            prev_month = month - 1 if month > 1 else 12
            yoy_year   = year - 1
            prev_vol   = self.monthly_totals_.get((prev_year,  prev_month), None)
            yoy_vol    = self.monthly_totals_.get((yoy_year,   month),      None)
            pp_vol     = self.monthly_totals_.get((prev_year,  month),      None)
            if prev_vol is None or yoy_vol is None or pp_vol is None:
                return 1.0
            if pp_vol <= 0 or prev_vol <= 0:
                return 1.0
            mom_trend = prev_vol / pp_vol
            weight    = 1.0 + self.mom_dampening * (mom_trend - 1.0)
            weight    = np.clip(weight, 0.85, 1.15)
            return float(weight)
        except Exception:
            return 1.0

In [17]:
#///////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////
#                                         Model C — StructuralLoanForecaster_C (December Specialist)                              #
#///////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////

class StructuralLoanForecaster_C:
    """
    December-specialist structural forecaster for daily loan volumes.

    December is uniquely difficult because:
      - Volume collapses sharply after ~Dec 18 as borrowers/staff go offline
      - Early December often sees a catch-up surge from November pipeline
      - The Christmas–New Year holiday cluster is tightly spaced, causing
        cascading distortion across nearly every business day in the final 2 weeks

    Design decisions vs Model A/B:
      - Seasonal effects (DOW, DOM) fitted on December data ONLY,
        using all available historical Decembers (full months only)
      - Separate pre/post cliff base levels to capture the Dec 18 volume cliff
      - Holiday distortion window widened (before=2, after=6) to capture
        the full Christmas–NYE cascade — Dec 24 and Dec 31 suppression
        is learned empirically from history via distortion factors rather
        than being hard-coded
      - distortion_max_factor widened to 1.4 — December is more volatile
      - distortion_min_obs lowered to 2 — fewer historical Decembers
      - Trend fitted on all production data (not December-only) so the
        long-run growth rate is informed by the full history
    """

    def __init__(
        self,
        target                   = 'total_loan_amount',
        recency_strength         = 3.0,
        dow_shrinkage            = 0.15,
        interaction_shrinkage    = 0.25,
        level_lookback_days      = 60,
        holiday_window_before    = 2,
        holiday_window_after     = 6,
        holiday_shrinkage        = 0.2,
        distortion_min_obs       = 2,
        distortion_max_factor    = 1.4,
        trend_dampening          = 0.80,
        forward_lift_daily       = 0.0,
        cliff_date               = 18,
        cliff_shrinkage          = 0.5,
    ):
        self.target                = target
        self.recency_strength      = recency_strength
        self.dow_shrinkage         = dow_shrinkage
        self.interaction_shrinkage = interaction_shrinkage
        self.level_lookback_days   = level_lookback_days
        self.holiday_window_before = holiday_window_before
        self.holiday_window_after  = holiday_window_after
        self.holiday_shrinkage     = holiday_shrinkage
        self.distortion_min_obs    = distortion_min_obs
        self.distortion_max_factor = distortion_max_factor
        self.trend_dampening       = trend_dampening
        self.forward_lift_daily    = forward_lift_daily
        self.cliff_date            = cliff_date
        self.cliff_shrinkage       = cliff_shrinkage

        self.master_calendar   = None
        self.distortion_table_ = {}
        self.fitted            = False

    # ------------------------------------------------------------------
    def _compute_expected(self, df, t_arr):
        trend = np.exp(self.trend_intercept + self.trend_slope * t_arr)
        dow_e = df['dow'].map(self.dow_effect).fillna(1.0).values
        dom_e = df['dom'].map(self.dom_effect).fillna(1.0).values
        interaction_e = np.array([
            self.interaction_effect.get((d, 12), 1.0)
            for d in df['dow']
        ])
        return self.base_level * trend * dow_e * dom_e * interaction_e

    # ------------------------------------------------------------------
    def fit(self, df, holiday_calendar=None):

        df = df.copy()
        df = df.sort_values('Calendar_Date').reset_index(drop=True)
        df['Calendar_Date'] = pd.to_datetime(df['Calendar_Date'])
        df['dow']           = df['Calendar_Date'].dt.dayofweek
        df['dom']           = df['Calendar_Date'].dt.day
        df['moy']           = df['Calendar_Date'].dt.month
        df['is_weekend']    = df['dow'].isin([5, 6])

        # ---- Holiday calendar ----
        if holiday_calendar is not None:
            calendar = holiday_calendar.copy()
            calendar['Calendar_Date'] = pd.to_datetime(calendar['Calendar_Date'])
            self.master_calendar = calendar
            df = df.merge(
                calendar[['Calendar_Date', 'Is_Holiday']],
                on='Calendar_Date', how='left'
            )
            df['Is_Holiday'] = df['Is_Holiday'].fillna(0)
        else:
            df['Is_Holiday'] = 0
            self.master_calendar = None

        df = df.reset_index(drop=True)

        # ---- Production mask (all data — used for trend) ----
        prod_mask = (
            (~df['is_weekend']) &
            (df['Is_Holiday'] == 0) &
            (df[self.target] > 0)
        )

        # ---- Holiday halo — exclude halo days from seasonal fitting ----
        if self.master_calendar is not None:
            holiday_dates = set(pd.to_datetime(
                self.master_calendar.loc[
                    self.master_calendar['Is_Holiday'] == 1, 'Calendar_Date'
                ]
            ))
            halo_dates = set()
            for hdate in holiday_dates:
                for offset in range(
                    -self.holiday_window_before,
                     self.holiday_window_after + 1
                ):
                    candidate = hdate + pd.Timedelta(days=offset)
                    if candidate.dayofweek not in [5, 6]:
                        halo_dates.add(candidate)
            df['in_halo'] = df['Calendar_Date'].isin(halo_dates)
        else:
            holiday_dates = set()
            df['in_halo']  = False

        # ---- December-only seasonal mask ----
        # Full completed Decembers only, halo days excluded so the
        # Christmas/NYE distortion doesn't corrupt DOW/DOM weights
        last_date           = df['Calendar_Date'].max()
        partial_month_start = last_date.replace(day=1)

        dec_seasonal_mask = (
            prod_mask &
            (df['moy'] == 12) &
            (~df['in_halo']) &
            (df['Calendar_Date'] < partial_month_start)
        )

        self.last_train_date = df['Calendar_Date'].max()
        self.n_train         = len(df)

        # ---- Trend — fitted on ALL production data ----
        df_prod      = df.loc[prod_mask].copy()
        t_prod       = np.arange(len(df_prod))
        recency_prod = np.exp(self.recency_strength * (t_prod / len(df_prod)))
        y_log        = np.log(df_prod[self.target].values)
        X            = np.vstack([np.ones(len(t_prod)), t_prod]).T
        W            = np.diag(recency_prod)
        beta         = np.linalg.inv(X.T @ W @ X) @ (X.T @ W @ y_log)
        self.trend_intercept = beta[0]
        self.trend_slope     = beta[1]

        t_full       = np.arange(len(df))
        trend_fitted = np.exp(self.trend_intercept + self.trend_slope * t_full)
        df['trend_fitted'] = trend_fitted
        df['detrended']    = np.where(
            trend_fitted > 0, df[self.target] / trend_fitted, 0.0
        )

        # ---- Base level ----
        recent_prod     = df.loc[prod_mask].tail(self.level_lookback_days)
        target_mean     = float(recent_prod[self.target].mean())
        trend_mean      = float(recent_prod['trend_fitted'].mean())
        self.base_level = target_mean / trend_mean if trend_mean > 0 else 1.0

        # ---- Pre/post cliff base levels ----
        # Measures how much volume drops after cliff_date in December
        # so the forecast can scale down the back half of the month
        dec_pre_mask  = dec_seasonal_mask & (df['dom'] <= self.cliff_date)
        dec_post_mask = dec_seasonal_mask & (df['dom'] >  self.cliff_date)

        pre_mean  = df.loc[dec_pre_mask,  self.target].mean()
        post_mean = df.loc[dec_post_mask, self.target].mean()

        if pd.notna(pre_mean) and pd.notna(post_mean) and pre_mean > 0:
            self.cliff_ratio = post_mean / pre_mean
        else:
            self.cliff_ratio = 1.0

        print(f"  [Model C] cliff_ratio (post/pre Dec {self.cliff_date}): "
              f"{self.cliff_ratio:.3f}")

        # ---- DOW effect — December data only, halo excluded ----
        dow_raw    = df.loc[dec_seasonal_mask].groupby('dow')['detrended'].mean()
        dow_shrunk = (
            (1 - self.dow_shrinkage) * dow_raw +
            self.dow_shrinkage * self.base_level
        )
        self.dow_effect        = dow_shrunk / dow_shrunk.mean()
        self.dow_effect.loc[5] = 0.0
        self.dow_effect.loc[6] = 0.0

        df['after_dow'] = df['detrended'] / df['dow'].map(
            lambda x: self.dow_effect.get(x, 1.0)
        )

        # ---- DOM effect — December data only, halo excluded ----
        dom_raw         = df.loc[dec_seasonal_mask].groupby('dom')['after_dow'].mean()
        self.dom_effect = dom_raw / dom_raw.mean()
        df['after_dom'] = df['after_dow'] / df['dom'].map(self.dom_effect)

        # ---- DOW x December interaction ----
        interaction_raw    = (
            df.loc[dec_seasonal_mask]
            .groupby(['dow', 'moy'])['after_dom']
            .mean()
        )
        interaction_shrunk = (
            (1 - self.interaction_shrinkage) * interaction_raw +
            self.interaction_shrinkage * interaction_raw.mean()
        )
        self.interaction_effect = interaction_shrunk / interaction_shrunk.mean()

        # ---- Holiday distortion ----
        # wider window (after=6) captures the full Christmas–NYE cascade.
        # Dec 24 / Dec 31 suppression is learned here from history as
        # distortion factors rather than being hard-coded.
        if self.master_calendar is not None:

            # Use only December holidays as anchors
            dec_holidays = [h for h in holiday_dates if h.month == 12]

            t_full_arr     = np.arange(len(df))
            df['expected'] = self._compute_expected(df, t_full_arr)

            biz_day_list = df.loc[
                ~df['is_weekend']
            ]['Calendar_Date'].sort_values().tolist()
            biz_day_pos  = {d: i for i, d in enumerate(biz_day_list)}

            ratio_accumulator = {}

            for hdate in dec_holidays:
                holiday_dow = hdate.dayofweek
                if holiday_dow in [5, 6]:
                    continue
                if hdate not in biz_day_pos:
                    continue
                h_pos = biz_day_pos[hdate]

                for offset in range(
                    -self.holiday_window_before,
                     self.holiday_window_after + 1
                ):
                    if offset == 0:
                        continue
                    tp = h_pos + offset
                    if tp < 0 or tp >= len(biz_day_list):
                        continue
                    target_date = biz_day_list[tp]
                    row         = df.loc[df['Calendar_Date'] == target_date]
                    if row.empty:
                        continue
                    if row['Is_Holiday'].values[0] == 1:
                        continue
                    if row['is_weekend'].values[0]:
                        continue
                    actual   = row[self.target].values[0]
                    expected = row['expected'].values[0]
                    if expected <= 0 or actual <= 0:
                        continue
                    ratio = actual / expected
                    ratio_accumulator.setdefault(
                        (holiday_dow, offset), []
                    ).append(ratio)

            self.distortion_table_ = {}
            for key, ratios in ratio_accumulator.items():
                if len(ratios) < self.distortion_min_obs:
                    continue
                raw_median = float(np.median(ratios))
                shrunk     = (
                    (1 - self.holiday_shrinkage) * raw_median +
                    self.holiday_shrinkage * 1.0
                )
                capped = float(np.clip(
                    shrunk,
                    1.0 / self.distortion_max_factor,
                    self.distortion_max_factor
                ))
                self.distortion_table_[key] = capped

            print(f"  [Model C] Distortion keys learned: "
                  f"{sorted(self.distortion_table_.keys())}")
        else:
            self.distortion_table_ = {}

        self.fitted = True
        return self

    # ------------------------------------------------------------------
    def forecast_from_date(self, start_date, months_forward=1):

        if not self.fitted:
            raise Exception('Model must be fit before forecasting.')

        start_date   = pd.to_datetime(start_date)
        end_date     = start_date + pd.DateOffset(months=months_forward)
        future_dates = pd.date_range(
            start=start_date,
            end=end_date - pd.Timedelta(days=1),
            freq='D'
        )

        df = pd.DataFrame({'Calendar_Date': future_dates})
        df['dow']        = df['Calendar_Date'].dt.dayofweek
        df['dom']        = df['Calendar_Date'].dt.day
        df['moy']        = df['Calendar_Date'].dt.month
        df['is_weekend'] = df['dow'].isin([5, 6])

        # ---- Trend ----
        t_future         = np.arange(self.n_train, self.n_train + len(df))
        trend_multiplier = np.exp(
            self.trend_intercept +
            (self.trend_slope * self.trend_dampening) * t_future
        )

        horizon_index = np.arange(len(df))
        forward_lift  = 1 + self.forward_lift_daily * horizon_index

        # ---- Seasonal ----
        dow_e = df['dow'].map(self.dow_effect).fillna(1.0).values
        dom_e = df['dom'].map(self.dom_effect).fillna(1.0).values
        interaction_e = np.array([
            self.interaction_effect.get((d, 12), 1.0)
            for d in df['dow']
        ])

        seasonal = dow_e * dom_e * interaction_e
        seasonal = seasonal / np.mean(seasonal)

        forecast = (
            self.base_level * seasonal * trend_multiplier * forward_lift
        )

        # ---- Pre/post cliff scaling ----
        post_cliff_mask = (
            (df['moy'] == 12) & (df['dom'] > self.cliff_date)
        ).values
        forecast[post_cliff_mask] = (
            forecast[post_cliff_mask] *
            (self.cliff_ratio + (1 - self.cliff_ratio) * self.cliff_shrinkage)
        )

        # ---- Weekend hard zero ----
        forecast[df['is_weekend'].values] = 0.0

        # ---- Holiday calendar ----
        if self.master_calendar is not None:
            df = df.merge(self.master_calendar, on='Calendar_Date', how='left')
            df['Is_Holiday'] = df['Is_Holiday'].fillna(0)

            # Hard zero on observed federal holidays (Dec 25, Jan 1, etc.)
            forecast[df['Is_Holiday'].values == 1] = 0.0

            # ---- Holiday distortion ----
            holiday_dates_forecast = self.master_calendar.loc[
                (self.master_calendar['Is_Holiday'] == 1) &
                (self.master_calendar['Calendar_Date'] >= start_date) &
                (self.master_calendar['Calendar_Date'] <= future_dates[-1])
            ]['Calendar_Date'].tolist()

            lookback_start = start_date - pd.Timedelta(
                days=(self.holiday_window_before + 1) * 2
            )
            holiday_dates_nearby = self.master_calendar.loc[
                (self.master_calendar['Is_Holiday'] == 1) &
                (self.master_calendar['Calendar_Date'] >= lookback_start) &
                (self.master_calendar['Calendar_Date'] < start_date)
            ]['Calendar_Date'].tolist()

            all_relevant_holidays = holiday_dates_nearby + holiday_dates_forecast
            distortion            = np.ones(len(df))

            for hdate in all_relevant_holidays:
                holiday_dow = hdate.dayofweek
                if holiday_dow in [5, 6]:
                    continue

                for offset in range(
                    -self.holiday_window_before,
                     self.holiday_window_after + 1
                ):
                    if offset == 0:
                        continue

                    candidate  = hdate
                    steps      = 0
                    direction  = 1 if offset > 0 else -1
                    abs_offset = abs(offset)
                    while steps < abs_offset:
                        candidate = candidate + pd.Timedelta(days=direction)
                        if candidate.dayofweek not in [5, 6]:
                            steps += 1

                    row_idx = df.index[df['Calendar_Date'] == candidate].tolist()
                    if not row_idx:
                        continue
                    idx    = row_idx[0]
                    factor = self.distortion_table_.get(
                        (holiday_dow, offset), 1.0
                    )
                    distortion[idx] = factor

            forecast = forecast * distortion

        df['forecast'] = forecast
        return df

In [19]:
class StructuralLoanForecaster_Switch:
    """
    Switching forecaster that selects Model A, B, or C
    based on the forecast month.

    Rules:
        Month == December              → Model C (December specialist)
        Month != December, >= 22 biz   → Model A
        Month != December, <= 21 biz   → Model B

    Handles partial months correctly: if the forecast starts mid-month,
    it still evaluates the FULL month's business day count to decide
    which model to use, then slices the forecast to only the relevant dates.
    """

    def __init__(self, today, target='total_loan_amount'):
        self.today            = pd.to_datetime(today)
        self.target           = target
        self.model_a          = None
        self.model_b          = None
        self.model_c          = None
        self.holiday_calendar = None
        self.fitted           = False

    @staticmethod
    def _count_biz_days(year, month, holiday_calendar):
        month_start = pd.Timestamp(year=year, month=month, day=1)
        month_end   = month_start + pd.offsets.MonthEnd(0)
        all_days    = pd.date_range(start=month_start, end=month_end, freq='D')
        holidays    = set(pd.to_datetime(
            holiday_calendar.loc[holiday_calendar['Is_Holiday'] == 1, 'Calendar_Date']
        ))
        biz_days = [
            d for d in all_days
            if d.dayofweek not in [5, 6] and d not in holidays
        ]
        return len(biz_days)

    def _select_model(self, year, month):
        if month == 12:
            print(
                f"  [Switch] {year}-{month:02d} → December → Model C selected"
            )
            return 'C', None

        n_biz = self._count_biz_days(year, month, self.holiday_calendar)
        model = 'A' if n_biz >= 22 else 'B'
        print(
            f"  [Switch] {year}-{month:02d} → {n_biz} business days → "
            f"Model {model} selected"
        )
        return model, n_biz

    def fit(self, df, holiday_calendar):
        self.holiday_calendar = holiday_calendar.copy()
        self.holiday_calendar['Calendar_Date'] = pd.to_datetime(
            self.holiday_calendar['Calendar_Date']
        )
        self.model_a = StructuralLoanForecaster_A()
        self.model_b = StructuralLoanForecaster_B()
        self.model_c = StructuralLoanForecaster_C()
        self.model_a.fit(df, holiday_calendar)
        self.model_b.fit(df, holiday_calendar)
        self.model_c.fit(df, holiday_calendar)
        self.fitted = True
        return self

    def forecast_from_date(self, start_date, months_forward=1):
        """
        Generates a forecast from start_date for months_forward months.

        For each calendar month touched by the forecast window:
          1. If December → Model C.
             Otherwise count full month business days → Model A or B.
          2. Run the chosen model for that month.
          3. Slice results to only the dates in [period_start, period_end].
        """
        if not self.fitted:
            raise Exception('Model must be fit before forecasting.')

        start_date = pd.to_datetime(start_date)
        records    = []

        for i in range(months_forward):
            if i == 0:
                period_start = start_date
            else:
                period_start = (start_date + pd.DateOffset(months=i)).replace(day=1)

            month_start = period_start.replace(day=1)
            period_end  = month_start + pd.offsets.MonthEnd(0)

            year  = month_start.year
            month = month_start.month

            model_choice, n_biz = self._select_model(year, month)

            if model_choice == 'A':
                model = self.model_a
            elif model_choice == 'B':
                model = self.model_b
            else:
                model = self.model_c

            pred = model.forecast_from_date(start_date=period_start, months_forward=1)
            pred = pred[
                (pred['Calendar_Date'] >= period_start) &
                (pred['Calendar_Date'] <= period_end)
            ].copy()

            pred['model_used'] = model_choice
            pred['n_biz_days'] = n_biz if n_biz is not None else self._count_biz_days(
                year, month, self.holiday_calendar
            )
            records.append(pred)

        return pd.concat(records, ignore_index=True)

In [20]:
#////////////////////////////////////////////////////////////////////////////////////////////////////////
#                                   BUSINESS DAY OFFSET HELPER
#////////////////////////////////////////////////////////////////////////////////////////////////////////

def add_business_days(start_date, n_days):
    current    = pd.to_datetime(start_date)
    days_added = 0
    while days_added < n_days:
        current += pd.Timedelta(days=1)
        if current.dayofweek not in [5, 6]:
            days_added += 1
    return current


#////////////////////////////////////////////////////////////////////////////////////////////////////////
#                                   COMPUTE KNOWN WINDOW & FORECAST START
#////////////////////////////////////////////////////////////////////////////////////////////////////////

KNOWN_THROUGH  = add_business_days(TODAY, 3)
FORECAST_START = add_business_days(TODAY, 4)
OUTPUT_THROUGH = add_business_days(TODAY, 29)

print(f"TODAY          : {TODAY.date()}")
print(f"Actuals through: {KNOWN_THROUGH.date()}")
print(f"Forecast start : {FORECAST_START.date()}")
print(f"Output through : {OUTPUT_THROUGH.date()}")


#////////////////////////////////////////////////////////////////////////////////////////////////////////
#                                   BUILD HOLIDAY CALENDAR
#////////////////////////////////////////////////////////////////////////////////////////////////////////

holiday_calendar = build_us_holiday_calendar(
    start_date='2021-01-01',
    end_date='2030-12-31'
)


#////////////////////////////////////////////////////////////////////////////////////////////////////////
#                                   TRAIN DATA (through KNOWN_THROUGH)
#////////////////////////////////////////////////////////////////////////////////////////////////////////

train_wholesale = wholesale_historical[
    (wholesale_historical['Calendar_Date'] >= '2021-09-01') &
    (wholesale_historical['Calendar_Date'] <= KNOWN_THROUGH)
].drop(columns=['Is_Holiday'], errors='ignore').copy()

train_wholesale['Calendar_Date'] = pd.to_datetime(train_wholesale['Calendar_Date'])


#////////////////////////////////////////////////////////////////////////////////////////////////////////
#                                   FIT SWITCH MODEL & FORECAST
#                                   months_forward=2 ensures we always cover a full
#                                   30-business-day window even when starting mid-month
#////////////////////////////////////////////////////////////////////////////////////////////////////////

model_switch = StructuralLoanForecaster_Switch(today=FORECAST_START)
model_switch.fit(train_wholesale, holiday_calendar)

pred_switch = model_switch.forecast_from_date(
    start_date=FORECAST_START,
    months_forward=2          # ← covers partial current month + next full month
)
pred_switch = pred_switch[pred_switch['Calendar_Date'] <= OUTPUT_THROUGH].copy()

#////////////////////////////////////////////////////////////////////////////////////////////////////////
#                                   PULL ACTUALS FOR KNOWN WINDOW
#////////////////////////////////////////////////////////////////////////////////////////////////////////

actuals_window = wholesale_historical[
    (wholesale_historical['Calendar_Date'] >= TODAY) &
    (wholesale_historical['Calendar_Date'] <= KNOWN_THROUGH)
][['Calendar_Date', 'total_loan_amount']].copy()

actuals_window['Calendar_Date'] = pd.to_datetime(actuals_window['Calendar_Date'])
actuals_window = actuals_window.rename(columns={'total_loan_amount': 'amount'})
actuals_window['source']     = 'actual'
actuals_window['model_used'] = '—'
actuals_window['forecast']   = None


#////////////////////////////////////////////////////////////////////////////////////////////////////////
#                                   BUILD COMBINED OUTPUT
#////////////////////////////////////////////////////////////////////////////////////////////////////////

forecast_out = pred_switch[['Calendar_Date', 'forecast', 'model_used']].copy()
forecast_out['amount'] = None
forecast_out['source'] = 'forecast'

wholesale_combined = pd.concat(
    [actuals_window, forecast_out], ignore_index=True
).sort_values('Calendar_Date').reset_index(drop=True)

wholesale_combined['channel'] = 'Wholesale'

# Business days only
wholesale_combined = wholesale_combined[
    ~wholesale_combined['Calendar_Date'].dt.dayofweek.isin([5, 6])
].reset_index(drop=True)


#////////////////////////////////////////////////////////////////////////////////////////////////////////
#                                   PRINT OUTPUT
#////////////////////////////////////////////////////////////////////////////////////////////////////////

dow_names = {0: 'Mon', 1: 'Tue', 2: 'Wed', 3: 'Thu', 4: 'Fri'}

def fmt(val):
    if val is None or pd.isna(val):
        return '         —'
    return f'{float(val):>12,.0f}'

header = (
    f"{'Date':<15} {'DOW':<6} {'Source':<10} {'Model':<8} "
    f"{'Channel':<14} {'Amount':>12}"
)
print(f"\n{header}")
print("-" * 70)

for _, row in wholesale_combined.iterrows():
    dow_label = dow_names.get(row['Calendar_Date'].dayofweek, '—')
    amount    = row['amount'] if row['source'] == 'actual' else row['forecast']
    print(
        f"{str(row['Calendar_Date'].date()):<15} "
        f"{dow_label:<6} "
        f"{row['source']:<10} "
        f"{str(row['model_used']):<8} "
        f"{row['channel']:<14} "
        f"{fmt(amount)}"
    )

print("-" * 70)

actual_total   = actuals_window['amount'].dropna().astype(float).sum()
forecast_total = pred_switch['forecast'].dropna().astype(float).sum()

print(
    f"{'TOTAL (actuals + forecast)':<46} "
    f"{fmt(actual_total + forecast_total)}"
)

TODAY          : 2026-03-10
Actuals through: 2026-03-13
Forecast start : 2026-03-16
Output through : 2026-04-20
  [Model C] cliff_ratio (post/pre Dec 18): 0.917
  [Model C] Distortion keys learned: [(0, -2), (0, -1), (0, 1), (0, 2), (0, 3), (0, 4), (0, 6), (4, -2), (4, -1), (4, 1), (4, 2), (4, 3), (4, 4), (4, 6)]
  [Switch] 2026-03 → 22 business days → Model A selected
  [Switch] 2026-04 → 22 business days → Model A selected

Date            DOW    Source     Model    Channel              Amount
----------------------------------------------------------------------
2026-03-10      Tue    actual     —        Wholesale         8,398,873
2026-03-11      Wed    actual     —        Wholesale        11,053,901
2026-03-12      Thu    actual     —        Wholesale         2,016,338
2026-03-13      Fri    actual     —        Wholesale         8,873,504
2026-03-16      Mon    forecast   A        Wholesale        14,764,677
2026-03-17      Tue    forecast   A        Wholesale         9,003,459
20

In [21]:
#////////////////////////////////////////////////////////////////////////////////////////////////////////
#                                   REBUILD EVAL_DF TO COVER FULL REQUESTED DATE RANGE
#////////////////////////////////////////////////////////////////////////////////////////////////////////

VIEW_FROM = pd.to_datetime(TODAY)
VIEW_TO   = pd.to_datetime('2026-04-10')

print(f"Requested range: {VIEW_FROM.date()} → {VIEW_TO.date()}")

# ── Pull actuals for the full requested window ────────────────────────────────
full_actuals = wholesale_historical[
    (wholesale_historical['Calendar_Date'] >= TODAY) &
    (wholesale_historical['Calendar_Date'] <= VIEW_TO)
][['Calendar_Date', 'total_loan_amount']].copy()

full_actuals['Calendar_Date'] = pd.to_datetime(full_actuals['Calendar_Date'])
full_actuals = full_actuals.rename(columns={'total_loan_amount': 'actual'})

# ── Re-fit switch model ─────────���─────────────────────────────────────────────
model_switch_eval = StructuralLoanForecaster_Switch(today=FORECAST_START)
model_switch_eval.fit(
    wholesale_historical[
        (wholesale_historical['Calendar_Date'] >= '2021-09-01') &
        (wholesale_historical['Calendar_Date'] <= KNOWN_THROUGH)
    ].drop(columns=['Is_Holiday'], errors='ignore').copy(),
    holiday_calendar
)

# months_forward=2 ensures we always cover VIEW_TO even for mid-month starts
months_needed = (
    (VIEW_TO.year - FORECAST_START.year) * 12 +
    (VIEW_TO.month - FORECAST_START.month) + 2
)

pred_switch_eval = model_switch_eval.forecast_from_date(
    start_date=FORECAST_START,
    months_forward=months_needed
)

# Cap to VIEW_TO
pred_switch_eval = pred_switch_eval[pred_switch_eval['Calendar_Date'] <= VIEW_TO].copy()

# ── Build eval_df ─────────────────────────────────────────────────────────────
eval_df = full_actuals.merge(
    pred_switch_eval[['Calendar_Date', 'forecast', 'model_used']],
    on='Calendar_Date', how='inner'
)

eval_df['channel'] = 'Wholesale'
eval_df['dow']     = eval_df['Calendar_Date'].dt.dayofweek

# Business days with actual > 0 only
eval_df = eval_df[
    (~eval_df['Calendar_Date'].dt.dayofweek.isin([5, 6])) &
    (eval_df['actual'] > 0)
].reset_index(drop=True)

# ── Known window: replace forecast with actuals ───────────────────────────────
known_mask = eval_df['Calendar_Date'] <= KNOWN_THROUGH
eval_df.loc[known_mask, 'forecast']   = eval_df.loc[known_mask, 'actual']
eval_df.loc[known_mask, 'model_used'] = 'actual'

eval_df['diff']    = eval_df['forecast'] - eval_df['actual']
eval_df['pct_err'] = (eval_df['diff'] / eval_df['actual']) * 100

eval_df['source'] = 'forecast'
eval_df.loc[known_mask, 'source'] = 'actual'

print(f"eval_df covers: {eval_df['Calendar_Date'].min().date()} → {eval_df['Calendar_Date'].max().date()}")
print(f"eval_df rows  : {len(eval_df)}")


#////////////////////////////////////////////////////////////////////////////////////////////////////////
#                                   PRINT OUTPUT TABLE
#////////////////////////////////////////////////////////////////////////////////////////////////////////

dow_names = {0: 'Mon', 1: 'Tue', 2: 'Wed', 3: 'Thu', 4: 'Fri'}

def fmt(val):
    if val is None or pd.isna(val):
        return '—'
    return f'{float(val):,.0f}'

def fmt_pct(val):
    if val is None or pd.isna(val):
        return '—'
    return f'{float(val):+.1f}%'

# ── Known actuals window ──────────────────────────────────────────────────────
known_actuals_print = wholesale_historical[
    (wholesale_historical['Calendar_Date'] >= TODAY) &
    (wholesale_historical['Calendar_Date'] <= KNOWN_THROUGH) &
    (~wholesale_historical['Calendar_Date'].dt.dayofweek.isin([5, 6])) &
    (wholesale_historical['total_loan_amount'] > 0)
][['Calendar_Date', 'total_loan_amount']].copy()

known_actuals_print['Calendar_Date'] = pd.to_datetime(known_actuals_print['Calendar_Date'])
known_actuals_print = known_actuals_print.rename(columns={'total_loan_amount': 'actual'})
known_actuals_print['source']    = 'actual'
known_actuals_print['channel']   = 'Wholesale'
known_actuals_print['forecast']  = known_actuals_print['actual']
known_actuals_print['model_used']= 'actual'
known_actuals_print['diff']      = 0.0
known_actuals_print['pct_err']   = 0.0

# ── Forecast rows ─────────────────────────────────────────────────────────────
forecast_rows = eval_df[eval_df['source'] == 'forecast'].copy()

# ── Combine ───────────────────────────────────────────────────────────────────
print_df = pd.concat(
    [known_actuals_print, forecast_rows], ignore_index=True
).sort_values(
    ['Calendar_Date', 'source'], ascending=[True, True]
).drop_duplicates(
    subset='Calendar_Date', keep='first'
).reset_index(drop=True)

# ── Apply date band filter ────────────────────────────────────────────────────
print_df_view = print_df[
    (print_df['Calendar_Date'] >= VIEW_FROM) &
    (print_df['Calendar_Date'] <= VIEW_TO)
].copy().reset_index(drop=True)

# ── Totals & MAE/MAPE ─────────────────────────────────────────────────────────
forecast_mask_view = print_df_view['source'] == 'forecast'

total_actual   = print_df_view.loc[forecast_mask_view, 'actual'].sum()
total_forecast = print_df_view.loc[forecast_mask_view, 'forecast'].sum()
total_diff     = total_forecast - total_actual
total_pct      = (total_diff / total_actual) * 100 if total_actual > 0 else float('nan')

mae  = print_df_view.loc[forecast_mask_view, 'diff'].abs().mean()    if forecast_mask_view.any() else float('nan')
mape = print_df_view.loc[forecast_mask_view, 'pct_err'].abs().mean() if forecast_mask_view.any() else float('nan')

# ── Build table columns ───────────────────────────────────────────────────────
col_date      = []
col_dow       = []
col_source    = []
col_model     = []
col_channel   = []
col_actual    = []
col_forecast  = []
col_diff      = []
col_pct       = []

for _, row in print_df_view.iterrows():
    col_date.append(str(row['Calendar_Date'].date()))
    col_dow.append(dow_names.get(row['Calendar_Date'].dayofweek, '—'))
    col_source.append(row['source'])
    col_model.append(str(row['model_used']))
    col_channel.append(row['channel'])
    col_actual.append(fmt(row['actual']))
    col_forecast.append(fmt(row['forecast']))
    col_diff.append(fmt(row['diff']))
    col_pct.append(fmt_pct(row['pct_err']))

# ── TOTAL row ─────────────────────────────────────────────────────────────────
col_date.append(f'TOTAL  ({VIEW_FROM.strftime("%b %d")} → {VIEW_TO.strftime("%b %d")})')
col_dow.append('')
col_source.append('forecast only')
col_model.append('')
col_channel.append('Wholesale')
col_actual.append(fmt(total_actual))
col_forecast.append(fmt(total_forecast))
col_diff.append(fmt(total_diff))
col_pct.append(fmt_pct(total_pct))

# ── MAE row ───────────────────────────────────────────────────────────────────
col_date.append('MAE')
for lst in [col_dow, col_source, col_model, col_channel, col_actual]:
    lst.append('')
col_forecast.append(fmt(mae))
col_diff.append('')
col_pct.append('')

# ── MAPE row ──────────────────────────────────────────────────────────────────
col_date.append('MAPE')
for lst in [col_dow, col_source, col_model, col_channel, col_actual]:
    lst.append('')
col_forecast.append(fmt_pct(mape))
col_diff.append('')
col_pct.append('')

# ── Row fill colours ──────────────────────────────────────────────────────────
n_known_view    = print_df_view[print_df_view['source'] == 'actual'].shape[0]
n_forecast_view = print_df_view[print_df_view['source'] == 'forecast'].shape[0]
n_summary       = 3

fill_colors = (
    ['#f0f0f0'] * n_known_view    +
    ['white']   * n_forecast_view +
    ['#fffbea'] * n_summary
)

# ── Plotly table ──────────────────────────────────────────────────────────────
table_fig = go.Figure(data=[go.Table(
    columnwidth = [110, 50, 80, 60, 80, 100, 100, 100, 80],
    header = dict(
        values = [
            '<b>Date</b>', '<b>DOW</b>', '<b>Source</b>', '<b>Model</b>',
            '<b>Channel</b>', '<b>Actual</b>',
            '<b>Forecast</b>', '<b>Diff</b>', '<b>Pct</b>'
        ],
        fill_color = '#2a3f5f',
        font       = dict(color='white', size=11),
        align      = ['left', 'center', 'center', 'center', 'center',
                      'right', 'right', 'right', 'right'],
        height     = 30
    ),
    cells = dict(
        values = [
            col_date, col_dow, col_source, col_model, col_channel,
            col_actual, col_forecast, col_diff, col_pct
        ],
        fill_color = [fill_colors] * 9,
        font       = dict(color='black', size=11),
        align      = ['left', 'center', 'center', 'center', 'center',
                      'right', 'right', 'right', 'right'],
        height     = 26
    )
)])

table_fig.update_layout(
    title = dict(
        text = (
            f'Wholesale — Actuals vs Forecast | '
            f'{VIEW_FROM.strftime("%b %d, %Y")} → {VIEW_TO.strftime("%b %d, %Y")}'
        ),
        font = dict(size=13)
    ),
    width  = 1000,
    height = 80 + (n_known_view + n_forecast_view + n_summary) * 28,
    margin = dict(l=10, r=10, t=50, b=10)
)

table_fig.show()


#////////////////////////////////////////////////////////////////////////////////////////////////////////
#                                   PLOT — ACTUALS vs FORECAST (SWITCH MODEL)
#////////////////////////////////////////////////////////////////////////////////////////////////////////

fig = go.Figure()

# ── Actuals line ──────────────────────────────────────────────────────────────
fig.add_trace(go.Scatter(
    x      = print_df['Calendar_Date'],
    y      = print_df['actual'],
    mode   = 'lines+markers',
    name   = 'Actual',
    line   = dict(color='black', width=2),
    marker = dict(size=6)
))

# ── Forecast line (colour-coded by model) ─────────────────────────────────────
forecast_only = print_df[print_df['source'] == 'forecast'].copy()

for model_label, color in [('A', 'steelblue'), ('B', 'darkorange')]:
    mask = forecast_only['model_used'] == model_label
    if mask.any():
        fig.add_trace(go.Scatter(
            x      = forecast_only.loc[mask, 'Calendar_Date'],
            y      = forecast_only.loc[mask, 'forecast'],
            mode   = 'lines+markers',
            name   = f'Forecast (Model {model_label})',
            line   = dict(color=color, width=2.5, dash='dash'),
            marker = dict(size=6)
        ))

fig.add_vline(
    x          = str(FORECAST_START.date()),
    line_width = 1.5,
    line_dash  = 'dash',
    line_color = 'grey'
)
fig.add_annotation(
    x=str(FORECAST_START.date()), y=1, yref='paper',
    text='Forecast Start', showarrow=False,
    xanchor='left', font=dict(color='grey', size=11)
)
fig.add_vrect(
    x0=str(TODAY.date()), x1=str(KNOWN_THROUGH.date()),
    fillcolor='lightgrey', opacity=0.2, layer='below', line_width=0
)
fig.add_annotation(
    x=str(TODAY.date()), y=1, yref='paper',
    text='Known Actuals', showarrow=False,
    xanchor='left', font=dict(color='grey', size=11)
)

fig.update_layout(
    title = dict(
        text=(
            f'Wholesale — Actuals vs Forecast (Switch Model)<br>'
            f'<sup>{TODAY.strftime("%b %d, %Y")} → {VIEW_TO.strftime("%b %d, %Y")}</sup>'
        ),
        font=dict(size=18)
    ),
    xaxis=dict(title='Date', tickformat='%b %d', tickangle=-45,
               showgrid=True, gridcolor='lightgrey'),
    yaxis=dict(title='Loan Volume ($)', tickformat='$,.0f',
               showgrid=True, gridcolor='lightgrey'),
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='left', x=0),
    hovermode='x unified', plot_bgcolor='white', width=1200, height=550
)

fig.show()

Requested range: 2026-03-10 → 2026-04-10
  [Model C] cliff_ratio (post/pre Dec 18): 0.917
  [Model C] Distortion keys learned: [(0, -2), (0, -1), (0, 1), (0, 2), (0, 3), (0, 4), (0, 6), (4, -2), (4, -1), (4, 1), (4, 2), (4, 3), (4, 4), (4, 6)]
  [Switch] 2026-03 → 22 business days → Model A selected
  [Switch] 2026-04 → 22 business days → Model A selected
  [Switch] 2026-05 → 20 business days → Model B selected
eval_df covers: 2026-03-16 → 2026-04-10
eval_df rows  : 20


Pipeline Backtesting

In [23]:
#////////////////////////////////////////////////////////////////////////////////////////////////////////
#                   STEP 1b: Cast Decimal columns to float
#////////////////////////////////////////////////////////////////////////////////////////////////////////

# SQL DECIMAL/NUMERIC types arrive as Python decimal.Decimal — pandas can't do arithmetic on them
decimal_cols = pipeline_wholesale.select_dtypes(include=['object']).columns.tolist()

# Force all numeric-looking columns to float
for col in pipeline_wholesale.columns:
    if col not in ['filedt', 'channel', 'year_month']:
        pipeline_wholesale[col] = pd.to_numeric(pipeline_wholesale[col], errors='coerce').astype(float)

print(f"Converted dtypes:\n{pipeline_wholesale.dtypes}")

Converted dtypes:
filedt                                datetime64[ns]
channel                                       object
loan_count                                   float64
loan_volume                                  float64
funded_loan_count                            float64
funded_loan_volume                           float64
underwriting_submission_count                float64
initial_conditional_approval_count           float64
underwriting_submission_events               float64
approval_events                              float64
application_count                            float64
pull_through_pct_count                       float64
pull_through_pct_volume                      float64
Biz_Day                                      float64
Biz_Day_Remaining_in_Month                   float64
Biz_Days_in_Month                            float64
Is_Holiday                                   float64
Is_Company_Holiday                           float64
Calendar_Days_Remaining     

In [24]:
#////////////////////////////////////////////////////////////////////////////////////////////////////////
#                   STEP 2: Aggregate to Monthly Level
#////////////////////////////////////////////////////////////////////////////////////////////////////////

pipeline_wholesale['year_month'] = pipeline_wholesale['filedt'].dt.to_period('M')

# ── Volume/count indicators — SUM monthly ────────────────────────────
sum_cols = [
    'loan_count',
    'loan_volume',
    'funded_loan_count',
    'funded_loan_volume',
    'underwriting_submission_count',
    'initial_conditional_approval_count',
    'underwriting_submission_events',
    'approval_events',
    'application_count',
]

monthly_sums = pipeline_wholesale.groupby('year_month')[sum_cols].sum().reset_index()

# ── Pull-through — recompute from monthly aggregates ─────────────────
monthly_sums['pull_through_pct_count'] = (
    100.0 * monthly_sums['funded_loan_count'] / monthly_sums['loan_count']
)
monthly_sums['pull_through_pct_volume'] = (
    100.0 * monthly_sums['funded_loan_volume'] / monthly_sums['loan_volume']
)

In [25]:
# ── Calendar/biz day indicators — aggregate monthly ──────────────────
#   Biz_Days_in_Month: constant within a month, take max
#   Biz_Day_Remaining_in_Month on last filedt of month: tells us coverage
#   Is_Holiday / Is_Company_Holiday: count how many holiday days in month
#   Calendar_Days_Remaining on last filedt: coverage check

monthly_cal = pipeline_wholesale.groupby('year_month').agg(
    biz_days_in_month=('Biz_Days_in_Month', 'max'),
    biz_days_with_data=('Biz_Day', 'sum'),               # how many biz days had pipeline data
    holiday_count=('Is_Holiday', 'sum'),                   # federal holidays in month
    company_holiday_count=('Is_Company_Holiday', 'sum'),   # company holidays in month
    min_biz_remaining=('Biz_Day_Remaining_in_Month', 'min'),  # should be 0 or 1 for complete months
    min_cal_remaining=('Calendar_Days_Remaining', 'min'),
).reset_index()

# Derived biz day features
monthly_cal['biz_day_coverage_pct'] = (
    monthly_cal['biz_days_with_data'] / monthly_cal['biz_days_in_month']
)
monthly_cal['effective_biz_days'] = (
    monthly_cal['biz_days_in_month'] - monthly_cal['holiday_count'] - monthly_cal['company_holiday_count']
)

# ── Merge sums + calendar ────────────────────────────────────────────
monthly_indicators = monthly_sums.merge(monthly_cal, on='year_month', how='left')
monthly_indicators['year_month_str'] = monthly_indicators['year_month'].astype(str)

# ── Per-biz-day normalized indicators ────────────────────────────────
#   Controls for months with different numbers of business days
for col in sum_cols:
    monthly_indicators[f'{col}_per_bizday'] = (
        monthly_indicators[col] / monthly_indicators['effective_biz_days']
    )

# Full list of indicator columns for correlation testing
all_indicator_cols = (
    sum_cols +
    ['pull_through_pct_count', 'pull_through_pct_volume'] +
    ['biz_days_in_month', 'holiday_count', 'company_holiday_count',
     'effective_biz_days', 'biz_day_coverage_pct'] +
    [f'{c}_per_bizday' for c in sum_cols]
)

print(f"\nMonthly rows: {len(monthly_indicators)}")
print(f"Indicator columns: {len(all_indicator_cols)}")
monthly_indicators[['year_month_str', 'biz_days_in_month', 'effective_biz_days',
                     'holiday_count', 'company_holiday_count']].tail(15)


Monthly rows: 26
Indicator columns: 25


,year_month_str,biz_days_in_month,effective_biz_days,holiday_count,company_holiday_count
11,2025-02,20.0,19.0,1.0,0.0
12,2025-03,21.0,21.0,0.0,0.0
13,2025-04,22.0,22.0,0.0,0.0
14,2025-05,21.0,19.0,1.0,1.0
15,2025-06,21.0,20.0,1.0,0.0
16,2025-07,22.0,20.0,1.0,1.0
17,2025-08,21.0,21.0,0.0,0.0
18,2025-09,21.0,19.0,1.0,1.0
19,2025-10,23.0,22.0,1.0,0.0
20,2025-11,19.0,16.0,2.0,1.0


In [26]:
#////////////////////////////////////////////////////////////////////////////////////////////////////////
#                   STEP 3: Compute Growth Rates — MoM, 2-Month, 3-Month
#////////////////////////////////////////////////////////////////////////////////////////////////////////

growth_cols_1m = []
growth_cols_2m = []
growth_cols_3m = []

for col in all_indicator_cols:
    g1 = f'{col}_growth_1m'
    monthly_indicators[g1] = monthly_indicators[col].pct_change()
    growth_cols_1m.append(g1)

    g2 = f'{col}_growth_2m'
    monthly_indicators[g2] = monthly_indicators[col] / monthly_indicators[col].shift(2) - 1
    growth_cols_2m.append(g2)

    g3 = f'{col}_growth_3m'
    monthly_indicators[g3] = monthly_indicators[col] / monthly_indicators[col].shift(3) - 1
    growth_cols_3m.append(g3)

all_growth_cols = growth_cols_1m + growth_cols_2m + growth_cols_3m

# Also include RAW levels (not just growth) — biz_days_in_month doesn't grow, it's structural
all_test_cols = all_indicator_cols + all_growth_cols

print(f"Total columns to test: {len(all_test_cols)}")

Total columns to test: 100


In [27]:
#////////////////////////////////////////////////////////////////////////////////////////////////////////
#                   STEP 4: Optimal Params from Backtest (param_grid_testing.txt)
#////////////////////////////////////////////////////////////////////////////////////////////////////////

optimal_params = pd.DataFrame([
    {'period': '2025-01', 'regime': 'January',    'recency_strength': 0.20, 'trend_dampening': 0.70, 'forward_lift_daily': 0.000, 'level_lookback_days': 60,  'trend_seasonal_tilt': 0.08, 'dow_shrinkage': 0.25, 'interaction_shrinkage': 0.20, 'seasonal_exponent': 1.05, 'actual_total': 165_761_576},
    {'period': '2025-02', 'regime': 'Regime1',    'recency_strength': 1.00, 'trend_dampening': 0.70, 'forward_lift_daily': 0.005, 'level_lookback_days': 90,  'trend_seasonal_tilt': 0.15, 'dow_shrinkage': 0.10, 'interaction_shrinkage': 0.35, 'seasonal_exponent': 1.05, 'actual_total': 175_734_244},
    {'period': '2025-03', 'regime': 'Regime1',    'recency_strength': 1.00, 'trend_dampening': 0.70, 'forward_lift_daily': 0.000, 'level_lookback_days': 90,  'trend_seasonal_tilt': 0.00, 'dow_shrinkage': 0.10, 'interaction_shrinkage': 0.20, 'seasonal_exponent': 1.05, 'actual_total': 181_623_181},
    {'period': '2025-04', 'regime': 'Regime1',    'recency_strength': 1.00, 'trend_dampening': 0.70, 'forward_lift_daily': 0.005, 'level_lookback_days': 90,  'trend_seasonal_tilt': 0.15, 'dow_shrinkage': 0.10, 'interaction_shrinkage': 0.20, 'seasonal_exponent': 1.00, 'actual_total': 194_492_552},
    {'period': '2025-05', 'regime': 'Regime1',    'recency_strength': 1.00, 'trend_dampening': 0.70, 'forward_lift_daily': 0.005, 'level_lookback_days': 90,  'trend_seasonal_tilt': 0.08, 'dow_shrinkage': 0.10, 'interaction_shrinkage': 0.20, 'seasonal_exponent': 1.00, 'actual_total': 174_537_762},
    {'period': '2025-06', 'regime': 'Regime1',    'recency_strength': 1.00, 'trend_dampening': 0.70, 'forward_lift_daily': 0.005, 'level_lookback_days': 90,  'trend_seasonal_tilt': 0.15, 'dow_shrinkage': 0.10, 'interaction_shrinkage': 0.20, 'seasonal_exponent': 1.05, 'actual_total': 179_300_643},
    {'period': '2025-07', 'regime': 'Regime1',    'recency_strength': 1.00, 'trend_dampening': 0.70, 'forward_lift_daily': 0.005, 'level_lookback_days': 90,  'trend_seasonal_tilt': 0.15, 'dow_shrinkage': 0.10, 'interaction_shrinkage': 0.50, 'seasonal_exponent': 1.05, 'actual_total': 193_088_074},
    {'period': '2025-08', 'regime': 'Regime1',    'recency_strength': 1.00, 'trend_dampening': 0.70, 'forward_lift_daily': 0.005, 'level_lookback_days': 90,  'trend_seasonal_tilt': 0.15, 'dow_shrinkage': 0.10, 'interaction_shrinkage': 0.20, 'seasonal_exponent': 1.05, 'actual_total': 199_246_002},
    {'period': '2025-09', 'regime': 'Transition', 'recency_strength': 1.00, 'trend_dampening': 0.90, 'forward_lift_daily': 0.000, 'level_lookback_days': 60,  'trend_seasonal_tilt': 0.00, 'dow_shrinkage': 0.25, 'interaction_shrinkage': 0.50, 'seasonal_exponent': 1.00, 'actual_total': 228_926_998},
    {'period': '2025-10', 'regime': 'Regime2',    'recency_strength': 0.35, 'trend_dampening': 0.83, 'forward_lift_daily': 0.000, 'level_lookback_days': 60,  'trend_seasonal_tilt': 0.08, 'dow_shrinkage': 0.05, 'interaction_shrinkage': 0.35, 'seasonal_exponent': 1.00, 'actual_total': 227_532_649},
    {'period': '2025-11', 'regime': 'Regime2',    'recency_strength': 0.35, 'trend_dampening': 0.83, 'forward_lift_daily': 0.000, 'level_lookback_days': 60,  'trend_seasonal_tilt': 0.00, 'dow_shrinkage': 0.25, 'interaction_shrinkage': 0.35, 'seasonal_exponent': 1.00, 'actual_total': 195_281_169},
    # Dec 2025 skipped — Model C
    {'period': '2026-01', 'regime': 'January',    'recency_strength': 0.20, 'trend_dampening': 0.70, 'forward_lift_daily': 0.005, 'level_lookback_days': 90,  'trend_seasonal_tilt': 0.00, 'dow_shrinkage': 0.25, 'interaction_shrinkage': 0.50, 'seasonal_exponent': 1.05, 'actual_total': 186_217_022},
    {'period': '2026-02', 'regime': 'Regime2',    'recency_strength': 1.00, 'trend_dampening': 0.83, 'forward_lift_daily': 0.000, 'level_lookback_days': 60,  'trend_seasonal_tilt': 0.00, 'dow_shrinkage': 0.25, 'interaction_shrinkage': 0.35, 'seasonal_exponent': 1.02, 'actual_total': 205_220_387},
    {'period': '2026-03', 'regime': 'Regime2',    'recency_strength': 1.00, 'trend_dampening': 0.83, 'forward_lift_daily': 0.000, 'level_lookback_days': 60,  'trend_seasonal_tilt': 0.08, 'dow_shrinkage': 0.05, 'interaction_shrinkage': 0.20, 'seasonal_exponent': 1.02, 'actual_total': 246_341_013},
])

regime_map = {'January': 0, 'Regime1': 1, 'Transition': 1.5, 'Regime2': 2}
optimal_params['regime_numeric'] = optimal_params['regime'].map(regime_map)
optimal_params['actual_total_growth'] = optimal_params['actual_total'].pct_change()

print(f"Optimal params rows: {len(optimal_params)}")
optimal_params[['period', 'regime', 'actual_total']].round(3)

Optimal params rows: 14


,period,regime,actual_total
0,2025-01,January,165761576
1,2025-02,Regime1,175734244
2,2025-03,Regime1,181623181
3,2025-04,Regime1,194492552
4,2025-05,Regime1,174537762
5,2025-06,Regime1,179300643
6,2025-07,Regime1,193088074
7,2025-08,Regime1,199246002
8,2025-09,Transition,228926998
9,2025-10,Regime2,227532649


In [28]:
#////////////////////////////////////////////////////////////////////////////////////////////////////////
#                   STEP 5: Join — Test Lead Times of 0, 1, 2, 3 Months
#////////////////////////////////////////////////////////////////////////////////////////////////////////

merged_frames = {}

for lead in [0, 1, 2, 3]:
    label = f'lead_{lead}m'

    indicators_shifted = monthly_indicators.copy()
    indicators_shifted['join_period'] = (
        indicators_shifted['year_month'] + lead
    ).astype(str)

    merged = optimal_params.merge(
        indicators_shifted,
        left_on='period',
        right_on='join_period',
        how='inner'
    )
    merged_frames[label] = merged
    print(f"{label}: {len(merged)} months matched")

lead_0m: 14 months matched
lead_1m: 14 months matched
lead_2m: 14 months matched
lead_3m: 14 months matched


In [29]:
#////////////////////////////////////////////////////////////////////////////////////////////////////////
#                   STEP 6: Correlation — ALL Indicators vs Regime & Key Params
#
#   Tests both RAW levels (e.g. biz_days_in_month = 22) and growth rates
#   This catches structural features like "22 biz day months need Regime 1"
#////////////////////////////////////////////////////////////////////////////////////////////////////////

target_params = [
    'regime_numeric',
    'trend_dampening',
    'forward_lift_daily',
    'level_lookback_days',
    'recency_strength',
    'trend_seasonal_tilt',
    'seasonal_exponent',
    'dow_shrinkage',
    'interaction_shrinkage',
]

print("\n" + "=" * 100)
print("  CORRELATION: Pipeline + Calendar Indicators → Optimal Model A Params")
print("=" * 100)

correlation_results = []

for lead_label, merged in merged_frames.items():
    for indicator in all_test_cols:
        if indicator not in merged.columns:
            continue
        for param in target_params:
            valid = merged[[indicator, param]].replace([np.inf, -np.inf], np.nan).dropna()
            if len(valid) < 4:
                continue
            # Check for zero variance
            if valid[indicator].std() == 0 or valid[param].std() == 0:
                continue
            corr = valid[indicator].corr(valid[param])
            if pd.notna(corr):
                correlation_results.append({
                    'lead': lead_label,
                    'indicator': indicator,
                    'param': param,
                    'correlation': round(corr, 3),
                    'abs_corr': round(abs(corr), 3),
                    'n_obs': len(valid),
                })

corr_df = pd.DataFrame(correlation_results)

if not corr_df.empty:

    # ── Top 50 overall ────────────────────────────────────────────────
    top_all = corr_df.sort_values('abs_corr', ascending=False).head(50)
    print(f"\n{'=' * 100}")
    print("  TOP 50 CORRELATIONS (all lead times, all indicators)")
    print(f"{'=' * 100}")
    print(top_all[['lead', 'indicator', 'param', 'correlation', 'n_obs']].to_string(index=False))

    # ── Best indicator per param ──────────────────────────────────────
    print(f"\n{'=' * 100}")
    print("  BEST INDICATOR PER MODEL PARAM")
    print(f"{'=' * 100}")
    for param in target_params:
        param_df = corr_df[corr_df['param'] == param].sort_values('abs_corr', ascending=False)
        if not param_df.empty:
            best = param_df.iloc[0]
            print(
                f"  {param:<28s} ← {best['indicator']:<50s} "
                f"r={best['correlation']:+.3f}  lead={best['lead']}  n={best['n_obs']}"
            )

    # ── Best for REGIME detection ─────────────────────────────────────
    print(f"\n{'=' * 100}")
    print("  BEST INDICATORS FOR REGIME DETECTION (regime_numeric)")
    print(f"{'=' * 100}")
    regime_df = corr_df[corr_df['param'] == 'regime_numeric'].sort_values('abs_corr', ascending=False).head(25)
    print(regime_df[['lead', 'indicator', 'correlation', 'n_obs']].to_string(index=False))

    # ── Biz day indicators specifically ───────────────────────────────
    print(f"\n{'=' * 100}")
    print("  CALENDAR / BIZ DAY INDICATORS vs REGIME")
    print(f"{'=' * 100}")
    cal_indicators = [c for c in corr_df['indicator'].unique()
                      if any(x in c for x in ['biz_day', 'holiday', 'effective', 'calendar', 'per_bizday'])]
    cal_regime = corr_df[
        (corr_df['param'] == 'regime_numeric') &
        (corr_df['indicator'].isin(cal_indicators))
    ].sort_values('abs_corr', ascending=False)
    if not cal_regime.empty:
        print(cal_regime[['lead', 'indicator', 'correlation', 'n_obs']].to_string(index=False))
    else:
        print("  No calendar indicator correlations found.")

else:
    print("\n  ⚠ No correlations computed — check pipeline data date coverage.")


  CORRELATION: Pipeline + Calendar Indicators → Optimal Model A Params

  TOP 50 CORRELATIONS (all lead times, all indicators)
   lead                                               indicator                 param  correlation  n_obs
lead_2m               underwriting_submission_events_per_bizday      recency_strength       -0.932     14
lead_3m                                       application_count      recency_strength       -0.907     14
lead_2m                              approval_events_per_bizday      recency_strength       -0.858     14
lead_0m                                 holiday_count_growth_1m      recency_strength       -0.851     11
lead_1m                         company_holiday_count_growth_3m        regime_numeric       -0.845      8
lead_0m                         company_holiday_count_growth_1m        regime_numeric       -0.837      7
lead_0m                                 holiday_count_growth_3m      recency_strength       -0.833     11
lead_1m                 

In [30]:
#////////////////////////////////////////////////////////////////////////////////////////////////////////
#                   STEP 7: Heatmap — Lead Time × Indicator vs regime_numeric
#////////////////////////////////////////////////////////////////////////////////////////////////////////

regime_corrs = corr_df[corr_df['param'] == 'regime_numeric'].copy()

if not regime_corrs.empty:
    pivot = regime_corrs.pivot_table(
        index='indicator', columns='lead', values='correlation'
    )
    pivot['max_abs'] = pivot.abs().max(axis=1)
    pivot = pivot.sort_values('max_abs', ascending=False).drop(columns='max_abs').head(25)

    fig_heatmap = go.Figure(data=go.Heatmap(
        z=pivot.values,
        x=pivot.columns.tolist(),
        y=pivot.index.tolist(),
        colorscale='RdBu_r',
        zmid=0, zmin=-1, zmax=1,
        text=pivot.round(2).values,
        texttemplate='%{text}',
        textfont=dict(size=10),
    ))
    fig_heatmap.update_layout(
        title='Correlation: Pipeline + Calendar Indicators → Regime (by lead time)',
        xaxis_title='Lead Time',
        yaxis_title='Indicator',
        height=700, width=1000,
        yaxis=dict(autorange='reversed'),
    )
    fig_heatmap.show()

In [31]:
#////////////////////////////////////////////////////////////////////////////////////////////////////////
#                   STEP 8: Time Series — Best Indicator vs Regime Over Time
#////////////////////////////////////////////////////////////////////////////////////////////////////////

if not regime_corrs.empty:
    best_row = regime_corrs.sort_values('abs_corr', ascending=False).iloc[0]
    best_indicator = best_row['indicator']
    best_lead = best_row['lead']
    best_corr = best_row['correlation']
    merged_best = merged_frames[best_lead]

    print(f"\n  Best regime predictor: {best_indicator} at {best_lead} (r={best_corr:+.3f})")

    fig_ts = go.Figure()

    fig_ts.add_trace(go.Scatter(
        x=merged_best['period'],
        y=merged_best[best_indicator],
        mode='lines+markers',
        name=best_indicator,
        yaxis='y1',
    ))

    fig_ts.add_trace(go.Scatter(
        x=merged_best['period'],
        y=merged_best['regime_numeric'],
        mode='lines+markers',
        name='Regime (0=Jan, 1=R1, 2=R2)',
        yaxis='y2',
        line=dict(dash='dot', color='red'),
    ))

    # Format y-axis based on whether best indicator is a growth rate or raw level
    is_growth = '_growth_' in best_indicator
    yaxis_fmt = '.0%' if is_growth else ',.0f'

    fig_ts.update_layout(
        title=f'Best Regime Predictor: {best_indicator} ({best_lead}, r={best_corr:+.3f})',
        xaxis_title='Forecast Month',
        yaxis=dict(title=best_indicator, tickformat=yaxis_fmt, side='left'),
        yaxis2=dict(title='Regime', overlaying='y', side='right', dtick=0.5),
        height=500, width=1100,
        hovermode='x unified',
        legend=dict(orientation='h', y=1.1),
    )
    fig_ts.show()


  Best regime predictor: company_holiday_count_growth_3m at lead_1m (r=-0.845)


In [32]:
#////////////////////////////////////////////////////////////////////////////////////////////////////////
#                   STEP 9: Scatter — Top Indicators vs Key Params
#////////////////////////////////////////////////////////////////////////////////////////////////////////

key_params_to_plot = [
    'trend_dampening', 'forward_lift_daily', 'level_lookback_days', 'regime_numeric'
]

if not regime_corrs.empty:
    top_6_indicators = (
        regime_corrs
        .sort_values('abs_corr', ascending=False)
        .drop_duplicates(subset='indicator')
        .head(6)
    )

    for param in key_params_to_plot:
        fig_scatter = go.Figure()
        for _, ind_row in top_6_indicators.iterrows():
            ind = ind_row['indicator']
            lead = ind_row['lead']
            m = merged_frames[lead]
            if ind not in m.columns:
                continue

            # Color by regime
            colors = m['regime'].map({
                'January': 'gold', 'Regime1': 'steelblue',
                'Transition': 'grey', 'Regime2': 'crimson'
            })

            is_growth = '_growth_' in ind
            fig_scatter.add_trace(go.Scatter(
                x=m[ind],
                y=m[param],
                mode='markers+text',
                text=m['period'],
                textposition='top center',
                name=f"{ind.split('_growth')[0]} [{lead}]",
                marker=dict(size=10, color=colors),
            ))

        xfmt = '.0%' if is_growth else ',.0f'
        fig_scatter.update_layout(
            title=f'{param} vs Top Pipeline/Calendar Indicators',
            xaxis_title='Indicator Value',
            xaxis_tickformat=xfmt,
            yaxis_title=param,
            height=450, width=1000,
        )
        fig_scatter.show()

In [33]:
#////////////////////////////////////////////////////////////////////////////////////////////////////////
#                   STEP 10: Per-Biz-Day Normalized View
#
#   This answers: "Is the regime shift driven by more volume per day
#   (true demand growth) or just more business days in the month?"
#////////////////////////////////////////////////////////////////////////////////////////////////////////

print(f"\n{'=' * 100}")
print("  PER-BIZ-DAY NORMALIZED INDICATORS vs REGIME")
print(f"{'=' * 100}")

per_bizday_cols = [c for c in corr_df['indicator'].unique() if 'per_bizday' in c]
per_bizday_regime = corr_df[
    (corr_df['param'] == 'regime_numeric') &
    (corr_df['indicator'].isin(per_bizday_cols))
].sort_values('abs_corr', ascending=False).head(15)

if not per_bizday_regime.empty:
    print(per_bizday_regime[['lead', 'indicator', 'correlation', 'n_obs']].to_string(index=False))
else:
    print("  No per-biz-day correlations found.")


  PER-BIZ-DAY NORMALIZED INDICATORS vs REGIME
   lead                                           indicator  correlation  n_obs
lead_1m                        application_count_per_bizday        0.743     14
lead_1m                approval_events_per_bizday_growth_1m        0.742     14
lead_1m underwriting_submission_events_per_bizday_growth_1m        0.702     14
lead_0m underwriting_submission_events_per_bizday_growth_2m        0.685     14
lead_0m                approval_events_per_bizday_growth_2m        0.683     14
lead_1m           underwriting_submission_events_per_bizday        0.643     14
lead_1m                          approval_events_per_bizday        0.640     14
lead_0m              application_count_per_bizday_growth_1m       -0.526     14
lead_1m              application_count_per_bizday_growth_1m        0.520     14
lead_0m                    loan_volume_per_bizday_growth_2m        0.493     14
lead_1m                    loan_volume_per_bizday_growth_1m        0.492 

In [34]:
#////////////////////////////////////////////////////////////////////////////////////////////////////////
#                   STEP 11: Summary
#////////////////////////////////////////////////////////////////////////////////////////////////////////

print("\n" + "=" * 100)
print("  NEXT STEPS")
print("=" * 100)
print("""
  1. HEATMAP (Step 7): Which indicators light up for regime_numeric?
     - Pipeline growth rates → demand-driven regime detection
     - Biz day counts       → structural/calendar-driven

  2. CALENDAR SECTION (Step 6 bottom): Does biz_days_in_month or
     effective_biz_days correlate with regime? If so, the current
     Switch model's biz-day logic has a basis.

  3. PER-BIZ-DAY (Step 10): If normalized indicators still correlate
     with regime, the shift is real demand growth — not just calendar.
     If only raw totals correlate, the "regime" might be a calendar artifact.

  4. SCATTER (Step 9): Look for clean thresholds:
     - "When pipeline_loan_volume_per_bizday_growth_2m > X%, use Regime 2"
     - "When effective_biz_days <= 20, use Model B params"

  5. Share the correlation tables and heatmap — we'll build the detector
     from whatever signals are strongest.
""")


  NEXT STEPS

  1. HEATMAP (Step 7): Which indicators light up for regime_numeric?
     - Pipeline growth rates → demand-driven regime detection
     - Biz day counts       → structural/calendar-driven

  2. CALENDAR SECTION (Step 6 bottom): Does biz_days_in_month or
     effective_biz_days correlate with regime? If so, the current
     Switch model's biz-day logic has a basis.

  3. PER-BIZ-DAY (Step 10): If normalized indicators still correlate
     with regime, the shift is real demand growth — not just calendar.
     If only raw totals correlate, the "regime" might be a calendar artifact.

  4. SCATTER (Step 9): Look for clean thresholds:
     - "When pipeline_loan_volume_per_bizday_growth_2m > X%, use Regime 2"
     - "When effective_biz_days <= 20, use Model B params"

  5. Share the correlation tables and heatmap — we'll build the detector
     from whatever signals are strongest.



In [35]:
#////////////////////////////////////////////////////////////////////////////////////////////////////////
#                   THRESHOLD ANALYSIS — Where do Regime 1 and Regime 2 separate?
#////////////////////////////////////////////////////////////////////////////////////////////////////////

merged_1m = merged_frames['lead_1m'].copy()

# The three indicators that matter
key_indicators = [
    'application_count_per_bizday',
    'approval_events_per_bizday_growth_1m',
    'underwriting_submission_events_per_bizday_growth_1m',
]

# ── Print raw values by regime ────────────────────────────────────────
print("=" * 100)
print("  RAW VALUES BY REGIME — lead_1m indicators")
print("=" * 100)

display_cols = ['period', 'regime', 'actual_total'] + key_indicators
print(merged_1m[display_cols].to_string(index=False))


# ── Stats by regime ──────────────────────────────────────────────────
print(f"\n{'=' * 100}")
print("  MEAN / MEDIAN BY REGIME")
print(f"{'=' * 100}")

for ind in key_indicators:
    print(f"\n  {ind}:")
    for regime in ['January', 'Regime1', 'Transition', 'Regime2']:
        subset = merged_1m.loc[merged_1m['regime'] == regime, ind]
        if not subset.empty:
            print(f"    {regime:<12s}  mean={subset.mean():>10.2f}  "
                  f"median={subset.median():>10.2f}  "
                  f"min={subset.min():>10.2f}  max={subset.max():>10.2f}  n={len(subset)}")


# ── Composite score ──────────────────────────────────────────────────
# Standardize each indicator, then average — single number for regime detection
print(f"\n{'=' * 100}")
print("  COMPOSITE SCORE (standardized average of 3 indicators)")
print(f"{'=' * 100}")

for ind in key_indicators:
    col_z = f'{ind}_z'
    mean_val = merged_1m[ind].mean()
    std_val = merged_1m[ind].std()
    merged_1m[col_z] = (merged_1m[ind] - mean_val) / std_val if std_val > 0 else 0

z_cols = [f'{ind}_z' for ind in key_indicators]
merged_1m['composite_score'] = merged_1m[z_cols].mean(axis=1)

# Correlation of composite with regime
composite_corr = merged_1m['composite_score'].corr(merged_1m['regime_numeric'])
print(f"\n  Composite score correlation with regime: r={composite_corr:+.3f}")

print(f"\n  {'Period':<10s} {'Regime':<12s} {'Composite':>10s}  {'Apps/BizDay':>12s}  {'Appr Growth':>12s}  {'UW Growth':>12s}")
print("  " + "-" * 80)

for _, row in merged_1m.sort_values('period').iterrows():
    print(f"  {row['period']:<10s} {row['regime']:<12s} "
          f"{row['composite_score']:>10.3f}  "
          f"{row['application_count_per_bizday']:>12.1f}  "
          f"{row.get('approval_events_per_bizday_growth_1m', float('nan')):>12.3f}  "
          f"{row.get('underwriting_submission_events_per_bizday_growth_1m', float('nan')):>12.3f}")


# ── Find optimal threshold ───────────────────────────────────────────
print(f"\n{'=' * 100}")
print("  THRESHOLD SEARCH — Composite Score → Regime Classification")
print(f"{'=' * 100}")

# Exclude January (handled separately)
non_jan = merged_1m[merged_1m['regime'] != 'January'].copy()
non_jan['is_regime2'] = (non_jan['regime'].isin(['Regime2', 'Transition'])).astype(int)

thresholds = np.arange(-1.5, 1.5, 0.1)
best_acc = 0
best_thresh = 0

print(f"\n  {'Threshold':>10s}  {'Accuracy':>10s}  {'Correct':>8s}  {'Total':>6s}")

for thresh in thresholds:
    predicted_r2 = (non_jan['composite_score'] >= thresh).astype(int)
    accuracy = (predicted_r2 == non_jan['is_regime2']).mean()
    correct = (predicted_r2 == non_jan['is_regime2']).sum()
    total = len(non_jan)

    if accuracy >= best_acc:
        best_acc = accuracy
        best_thresh = thresh

    if accuracy >= 0.75:
        print(f"  {thresh:>10.2f}  {accuracy:>10.1%}  {correct:>8d}  {total:>6d}")

print(f"\n  BEST THRESHOLD: {best_thresh:.2f}  →  Accuracy: {best_acc:.1%}")


# ── Classification table at best threshold ────────────────────────────
print(f"\n{'=' * 100}")
print(f"  CLASSIFICATION AT THRESHOLD = {best_thresh:.2f}")
print(f"{'=' * 100}")

merged_1m['predicted_regime'] = 'Regime1'  # default
merged_1m.loc[merged_1m['regime'] == 'January', 'predicted_regime'] = 'January'
merged_1m.loc[
    (merged_1m['regime'] != 'January') &
    (merged_1m['composite_score'] >= best_thresh),
    'predicted_regime'
] = 'Regime2'

print(f"\n  {'Period':<10s} {'Actual':<12s} {'Predicted':<12s} {'Composite':>10s} {'Match':>6s}")
print("  " + "-" * 60)

correct = 0
total = len(merged_1m)

for _, row in merged_1m.sort_values('period').iterrows():
    match = '✓' if row['regime'] == row['predicted_regime'] else '✗'
    if row['regime'] == row['predicted_regime']:
        correct += 1
    print(f"  {row['period']:<10s} {row['regime']:<12s} {row['predicted_regime']:<12s} "
          f"{row['composite_score']:>10.3f} {match:>6s}")

print(f"\n  Overall accuracy: {correct}/{total} = {correct/total:.1%}")


# ── Scatter: Composite Score vs Regime ────────────────────────────────
fig = go.Figure()

regime_colors = {
    'January': 'gold', 'Regime1': 'steelblue',
    'Transition': 'grey', 'Regime2': 'crimson'
}

for regime in ['January', 'Regime1', 'Transition', 'Regime2']:
    mask = merged_1m['regime'] == regime
    if mask.any():
        fig.add_trace(go.Scatter(
            x=merged_1m.loc[mask, 'composite_score'],
            y=merged_1m.loc[mask, 'regime_numeric'],
            mode='markers+text',
            text=merged_1m.loc[mask, 'period'],
            textposition='top center',
            name=regime,
            marker=dict(size=12, color=regime_colors[regime]),
        ))

fig.add_vline(
    x=best_thresh, line_dash='dash', line_color='red',
    annotation_text=f'Threshold = {best_thresh:.2f}',
    annotation_position='top right',
)

fig.update_layout(
    title='Composite Pipeline Score vs Regime (lead_1m)',
    xaxis_title='Composite Score (standardized)',
    yaxis_title='Regime (0=Jan, 1=R1, 1.5=Trans, 2=R2)',
    height=500, width=900,
)
fig.show()

  RAW VALUES BY REGIME — lead_1m indicators
 period     regime  actual_total  application_count_per_bizday  approval_events_per_bizday_growth_1m  underwriting_submission_events_per_bizday_growth_1m
2025-01    January     165761576                    153.263158                             -0.199699                                            -0.233992
2025-02    Regime1     175734244                    175.000000                             -0.036500                                             0.044321
2025-03    Regime1     181623181                    164.473684                              0.002076                                            -0.042882
2025-04    Regime1     194492552                    163.428571                             -0.069468                                            -0.020433
2025-05    Regime1     174537762                    148.363636                              0.088643                                            -0.016541
2025-06    Regime1     179300643